# US County Chronic Disease Burden (OKN)

- **Date:** 2026-11-18
- **Model:** claude-opus-5-5
- **Skills:** okn-report-style v0.1.7
- **SPARQL endpoint:** https://apps.okn.us/federation/sparql
- **Generated by:** mcp-okn v0.6.3 (build 2a4a1ef)
- **Queries:** 11
- **Elapsed time:** 2026-11-18 23:40–23:59 UTC (19m 3s)
- **Knowledge graphs used:** `spoke-okn`, `ruralkg`

## Prompt

Identify US counties with high prevalence of common chronic diseases, including diabetes, obesity, asthma, COPD, cardiovascular disease, hypertension, and other conditions supported by available knowledge graphs.

1. Discover data: Identify relevant knowledge graphs, retrieve county-level disease prevalence estimates, and document data sources, years, and coverage.
2. Analyze prevalence: Identify counties with the highest prevalence of individual diseases and those with high prevalence across multiple conditions. Develop a composite disease-burden score.
3. Create interactive maps: Visualize county-level disease prevalence and composite disease burden, with options to select different diseases.
4. Analyze correlations: Create a clustered correlation heatmap across diseases. Identify notable associations and consider potential demographic or geographic confounders.
5. Ensure reproducibility: Generate a complete, executable Jupyter Notebook containing the queries, data processing, analyses, visualizations, and key findings.

Use appropriate statistical methods, validate the data, and clearly document assumptions and limitations. Prioritize data available through mcp-okn and identify any gaps.

## Notes

County-level chronic-disease prevalence from spoke-okn (v0.0.6): native county estimates (County Health Rankings 2023; AHRQ SDOH 2019 CDC PLACES county measures) plus CDC PLACES place-level estimates rolled up to counties via spoke-okn PARTOF_LpL (population-weighted). Urbanicity (RUCC) from ruralkg (v0.2.10), joined on county FIPS. Composite = mean z-score over 11 conditions; correlations are Spearman, partial correlations from rank residuals after demographic/SES, state fixed-effect and behavioural adjustment; spatial clustering by Moran's I.

## Re-running this study

Every query cell loads the rows the session logged (`data/`), so the notebook runs offline and shows the original results. Set `RERUN = True` in the next cell (or start Jupyter with `OKN_RERUN=1`) to re-execute every query live against the OKN federation instead, or run the last cell (`verify_all()`) to compare each query's live row count with the logged one. A changed count usually means the knowledge graph was redeployed since the study ran, not that the analysis is wrong.

In [1]:
import os

import pandas as pd

from okn_sparql import run_query, verify_all

# False: load the rows logged during the session (data/). True: re-run each query live.
# Starting Jupyter with OKN_RERUN=1 sets it without editing the notebook.
RERUN = os.environ.get("OKN_RERUN", "") == "1"
data: dict[str, pd.DataFrame] = {}

## SPARQL queries

### q01 — Q1 - Inventory: which sources / geography levels carry disease prevalence in spoke-okn

2026-11-18T23:46:02+00:00 · `spoke-okn` · 2 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
# Q1 - Inventory of disease-prevalence statements in spoke-okn: source, geography level, measure type
SELECT ?source ?location_kind ?data_value_type (COUNT(DISTINCT ?disease) AS ?n_diseases) (COUNT(?stmt) AS ?n_statements) (COUNT(DISTINCT ?loc) AS ?n_locations) (SAMPLE(?yr) AS ?year) WHERE {
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:predicate kg:PREVALENCE_DpL ; rdf:subject ?disease ; rdf:object ?loc .
    OPTIONAL { ?stmt dcterms:source ?source }
    OPTIONAL { ?stmt kg:data_value_type ?data_value_type }
    OPTIONAL { ?stmt kg:year ?yr }
    OPTIONAL { ?loc <http://www.w3.org/1999/02/22-rdf-syntax-ns#> ?location_kind }
  }
} GROUP BY ?source ?location_kind ?data_value_type
```

In [2]:
data["q01"] = run_query("q01", rerun=RERUN)
data["q01"].head(10)

,source,location_kind,data_value_type,n_diseases,n_statements,n_locations,year
0,IHME,https://schema.org/Country,NaN,144,28800,200,2019.0
1,PLACES,https://schema.org/City,Age-adjusted prevalence,9,246285,27365,NaN


### q02 — Q2 - The nine CDC PLACES measures stored at place (City) level, and the DOID term each is mapped to

2026-11-18T23:46:05+00:00 · `spoke-okn` · 9 rows logged

```sparql
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
# Q2 - The nine CDC PLACES measures stored as place-level (City) disease prevalence, with the survey item each DOID term stands for
SELECT ?disease ?disease_label ?short_question_text ?data_value_type ?unit (COUNT(?stmt) AS ?n_places) (SUM(xsd:double(?tp)) AS ?pop_covered) WHERE {
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:predicate kg:PREVALENCE_DpL ; rdf:subject ?disease ; dcterms:source "PLACES" ;
          kg:short_question_text ?short_question_text ; kg:data_value_type ?data_value_type ; kg:data_value_unit ?unit ; kg:total_population ?tp .
    OPTIONAL { ?disease rdfs:label ?disease_label }
  }
} GROUP BY ?disease ?disease_label ?short_question_text ?data_value_type ?unit ORDER BY ?short_question_text
```

In [3]:
data["q02"] = run_query("q02", rerun=RERUN)
data["q02"].head(10)

,disease,disease_label,short_question_text,data_value_type,unit,n_places,pop_covered
0,http://purl.obolibrary.org/obo/DOID_3083,chronic obstructive pulmonary disease,COPD,Age-adjusted prevalence,%,27365,214016244.0
1,http://purl.obolibrary.org/obo/DOID_3393,coronary artery disease,Coronary Heart Disease,Age-adjusted prevalence,%,27365,214016244.0
2,http://purl.obolibrary.org/obo/DOID_2841,asthma,Current Asthma,Age-adjusted prevalence,%,27365,214016244.0
3,http://purl.obolibrary.org/obo/DOID_1596,depressive disorder,Depression,Age-adjusted prevalence,%,27365,214016244.0
4,http://purl.obolibrary.org/obo/DOID_9351,diabetes mellitus,Diabetes,Age-adjusted prevalence,%,27365,214016244.0
5,http://purl.obolibrary.org/obo/DOID_10763,hypertension,High Blood Pressure,Age-adjusted prevalence,%,27365,214016244.0
6,http://purl.obolibrary.org/obo/DOID_2349,arteriosclerosis,High Cholesterol,Age-adjusted prevalence,%,27365,214016244.0
7,http://purl.obolibrary.org/obo/DOID_9970,obesity,Obesity,Age-adjusted prevalence,%,27365,214016244.0
8,http://purl.obolibrary.org/obo/DOID_6713,cerebrovascular disease,Stroke,Age-adjusted prevalence,%,27365,214016244.0


### q03 — Q3 - Coverage, year and range of the native county measures and covariates

2026-11-18T23:46:29+00:00 · `spoke-okn` · 24 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX sd: <https://purl.org/okn/frink/kg/spoke-okn/sdoh/>
# Q3 - County-level (native) health and covariate measures in spoke-okn (SDoH PREVALENCEIN_SpL): source, year, coverage, range
SELECT ?col ?sdoh ?source ?yr (COUNT(DISTINCT ?loc) AS ?n_counties) (MIN(?num) AS ?min) (ROUND(100*AVG(?num))/100 AS ?mean) (MAX(?num) AS ?max) WHERE {
  VALUES (?sdoh ?col ?yr) {
    (sd:CHR_diabetes_prevalence "diabetes" "2023") (sd:414916001 "obesity" "2023") (sd:CDCP_ASTHMA_ADULT_A "asthma" "2019")
    (sd:CDCP_PULMONARY_ADULT_A "copd" "2019") (sd:CDCP_ARTHRITIS_ADULT_A "arthritis" "2019") (sd:CDCP_KIDNEY_DISEASE_ADULT_A "ckd" "2019")
    (sd:CDCP_CHOLES_ADULT_A "high_chol" "2019") (sd:CHR_frequent_mental_distress "mental_distress" "2023") (sd:CHR_frequent_physical_distress "physical_distress" "2023")
    (sd:CDCA_HEART_DTH_RATE_ABOVE35 "heart_mort" "2019") (sd:CDCA_STROKE_DTH_RATE_ABOVE35 "stroke_mort" "2019")
    (sd:CDCA_HEART_HOSP_RATE_ABOVE65 "heart_hosp" "2018") (sd:CDCA_STROKE_HOSP_RATE_ABOVE65 "stroke_hosp" "2018")
    (sd:ACS_TOT_POP_WT "pop" "2020") (sd:ACS_PCT_AGE_ABOVE65 "age65" "2020") (sd:ACS_PCT_BLACK_NONHISP "black" "2020")
    (sd:ACS_PCT_HISPANIC "hispanic" "2020") (sd:ACS_PCT_BACHELOR_DGR "bachelor" "2020") (sd:ACS_MEDIAN_HH_INC "income" "2020")
    (sd:SAIPE_PCT_POV "poverty" "2020") (sd:CHR_rural "rural" "2023") (sd:77176002 "smoking" "2023")
    (sd:CDCP_NO_PHY_ACTV_ADULT_A "inactivity" "2019") (sd:CHR_food_environment_index "food_env" "2023") }
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:subject ?sdoh ; rdf:predicate kg:PREVALENCEIN_SpL ; rdf:object ?loc ; kg:value ?val ; kg:year ?y ; dcterms:source ?source .
  }
  FILTER(STR(?y) = ?yr)
  BIND(STRAFTER(STR(?loc), "/location/") AS ?fips)
  FILTER(REGEX(?fips, "^[0-5][0-9]{4}$"))
  BIND(xsd:double(IF(CONTAINS(STR(?val),"("), STRBEFORE(STR(?val),"("), STR(?val))) AS ?num)
} GROUP BY ?col ?sdoh ?source ?yr ORDER BY ?source ?col
```

In [4]:
data["q03"] = run_query("q03", rerun=RERUN)
data["q03"].head(10)

,col,sdoh,source,yr,n_counties,min,mean,max
0,age65,https://purl.org/okn/frink/kg/spoke-okn/sdoh/A...,AHRQ SDOH Database,2020,3130,3.03,19.25,57.78
1,arthritis,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2019,3110,15.20,25.09,38.70
2,asthma,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2019,3110,7.20,9.72,13.30
3,bachelor,https://purl.org/okn/frink/kg/spoke-okn/sdoh/A...,AHRQ SDOH Database,2020,3130,0.00,14.63,43.00
4,black,https://purl.org/okn/frink/kg/spoke-okn/sdoh/A...,AHRQ SDOH Database,2020,3130,0.00,8.86,87.79
5,ckd,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2019,3110,2.00,2.94,5.90
6,copd,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2019,3110,3.20,7.40,15.50
7,heart_hosp,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2018,3114,19.10,60.05,126.50
8,heart_mort,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2019,3128,110.10,461.87,974.70
9,high_chol,https://purl.org/okn/frink/kg/spoke-okn/sdoh/C...,AHRQ SDOH Database,2019,3110,22.30,29.53,36.90


### q04 — Q4 - Native county measures + RUCC (ruralkg), state FIPS 01-13

2026-11-18T23:46:53+00:00 · `spoke-okn`, `ruralkg` · 544 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX sd: <https://purl.org/okn/frink/kg/spoke-okn/sdoh/>
PREFIX rst: <http://sail.ua.edu/ruralkg/settlementtype/>
# Q4 - Native county measures (spoke-okn: CHR 2023, AHRQ SDOH 2018-2020) pivoted one row per county + RUCC (ruralkg); state FIPS 01-13
SELECT ?fips ?county ?state ?rucc ?diabetes ?obesity ?asthma ?copd ?arthritis ?ckd ?high_chol ?mental_distress ?physical_distress ?heart_mort ?stroke_mort ?heart_hosp ?stroke_hosp ?pop ?age65 ?black ?hispanic ?bachelor ?income ?poverty ?rural ?smoking ?inactivity ?food_env WHERE {
 { SELECT ?fips (SAMPLE(?cn) AS ?county) (SAMPLE(?sn) AS ?state)
    (MAX(IF(?col="diabetes",?num,?u)) AS ?diabetes) (MAX(IF(?col="obesity",?num,?u)) AS ?obesity) (MAX(IF(?col="asthma",?num,?u)) AS ?asthma)
    (MAX(IF(?col="copd",?num,?u)) AS ?copd) (MAX(IF(?col="arthritis",?num,?u)) AS ?arthritis) (MAX(IF(?col="ckd",?num,?u)) AS ?ckd)
    (MAX(IF(?col="high_chol",?num,?u)) AS ?high_chol) (MAX(IF(?col="mental_distress",?num,?u)) AS ?mental_distress) (MAX(IF(?col="physical_distress",?num,?u)) AS ?physical_distress)
    (MAX(IF(?col="heart_mort",?num,?u)) AS ?heart_mort) (MAX(IF(?col="stroke_mort",?num,?u)) AS ?stroke_mort) (MAX(IF(?col="heart_hosp",?num,?u)) AS ?heart_hosp) (MAX(IF(?col="stroke_hosp",?num,?u)) AS ?stroke_hosp)
    (MAX(IF(?col="pop",?num,?u)) AS ?pop) (MAX(IF(?col="age65",?num,?u)) AS ?age65) (MAX(IF(?col="black",?num,?u)) AS ?black) (MAX(IF(?col="hispanic",?num,?u)) AS ?hispanic)
    (MAX(IF(?col="bachelor",?num,?u)) AS ?bachelor) (MAX(IF(?col="income",?num,?u)) AS ?income) (MAX(IF(?col="poverty",?num,?u)) AS ?poverty) (MAX(IF(?col="rural",?num,?u)) AS ?rural)
    (MAX(IF(?col="smoking",?num,?u)) AS ?smoking) (MAX(IF(?col="inactivity",?num,?u)) AS ?inactivity) (MAX(IF(?col="food_env",?num,?u)) AS ?food_env)
   WHERE {
    VALUES (?sdoh ?col ?yr) {
      (sd:CHR_diabetes_prevalence "diabetes" "2023") (sd:414916001 "obesity" "2023") (sd:CDCP_ASTHMA_ADULT_A "asthma" "2019")
      (sd:CDCP_PULMONARY_ADULT_A "copd" "2019") (sd:CDCP_ARTHRITIS_ADULT_A "arthritis" "2019") (sd:CDCP_KIDNEY_DISEASE_ADULT_A "ckd" "2019")
      (sd:CDCP_CHOLES_ADULT_A "high_chol" "2019") (sd:CHR_frequent_mental_distress "mental_distress" "2023") (sd:CHR_frequent_physical_distress "physical_distress" "2023")
      (sd:CDCA_HEART_DTH_RATE_ABOVE35 "heart_mort" "2019") (sd:CDCA_STROKE_DTH_RATE_ABOVE35 "stroke_mort" "2019")
      (sd:CDCA_HEART_HOSP_RATE_ABOVE65 "heart_hosp" "2018") (sd:CDCA_STROKE_HOSP_RATE_ABOVE65 "stroke_hosp" "2018")
      (sd:ACS_TOT_POP_WT "pop" "2020") (sd:ACS_PCT_AGE_ABOVE65 "age65" "2020") (sd:ACS_PCT_BLACK_NONHISP "black" "2020")
      (sd:ACS_PCT_HISPANIC "hispanic" "2020") (sd:ACS_PCT_BACHELOR_DGR "bachelor" "2020") (sd:ACS_MEDIAN_HH_INC "income" "2020")
      (sd:SAIPE_PCT_POV "poverty" "2020") (sd:CHR_rural "rural" "2023") (sd:77176002 "smoking" "2023")
      (sd:CDCP_NO_PHY_ACTV_ADULT_A "inactivity" "2019") (sd:CHR_food_environment_index "food_env" "2023") }
    GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?stmt rdf:subject ?sdoh ; rdf:predicate kg:PREVALENCEIN_SpL ; rdf:object ?loc ; kg:value ?val ; kg:year ?y .
      ?loc rdfs:label ?cn ; kg:state_name ?sn .
    }
    FILTER(STR(?y) = ?yr)
    BIND(STRAFTER(STR(?loc), "/location/") AS ?fips)
    FILTER(REGEX(?fips, "^(0[1-9]|1[0-3])[0-9]{3}$"))
    BIND(xsd:double(IF(CONTAINS(STR(?val),"("), STRBEFORE(STR(?val),"("), STR(?val))) AS ?num)
   } GROUP BY ?fips }
 BIND(IRI(CONCAT("http://stko-kwg.geog.ucsb.edu/lod/resource/administrativeRegion.USA.", ?fips)) AS ?reg)
 OPTIONAL { GRAPH <https://purl.org/okn/frink/kg/ruralkg> { ?cs rst:censusCounty ?reg ; rst:hasRUCC ?r . } }
 BIND(REPLACE(STR(?r), "^.*[^0-9]([0-9]+)$", "$1") AS ?rucc)
} ORDER BY ?fips
```

In [5]:
data["q04"] = run_query("q04", rerun=RERUN)
data["q04"].head(10)

,fips,county,state,rucc,diabetes,obesity,asthma,copd,arthritis,ckd,...,age65,black,hispanic,bachelor,income,poverty,rural,smoking,inactivity,food_env
0,1001,Autauga County,Alabama,2,10.6,37.3,9.7,7.5,29.8,2.8,...,15.26,19.50,2.88,16.69,57982.0,11.2,42.0022,18.3,32.2,6.6
1,1003,Baldwin County,Alabama,3,9.7,32.6,8.9,6.9,27.5,2.6,...,20.48,8.72,4.56,20.21,61756.0,8.9,42.2791,16.9,27.6,7.5
2,1005,Barbour County,Alabama,6,15.6,46.4,11.1,10.7,32.5,4.0,...,19.09,47.51,4.44,7.28,34990.0,25.5,67.7896,25.9,41.6,5.8
3,1007,Bibb County,Alabama,1,12.2,38.4,9.9,9.2,32.0,3.1,...,16.43,22.22,2.68,7.40,51721.0,17.8,68.3526,22.8,37.7,7.4
4,1009,Blount County,Alabama,1,10.9,32.7,9.7,8.7,30.7,2.9,...,17.98,1.33,9.28,8.89,48922.0,13.1,89.9515,21.8,33.3,7.8
5,1011,Bullock County,Alabama,6,18.1,48.3,10.8,9.2,31.6,4.1,...,16.10,68.61,8.10,6.09,33866.0,30.8,51.3744,25.5,41.3,4.3
6,1013,Butler County,Alabama,6,13.8,45.4,10.9,9.3,32.8,3.6,...,20.74,44.47,1.47,10.10,44850.0,20.6,71.2322,22.3,40.7,6.8
7,1015,Calhoun County,Alabama,3,12.3,37.4,10.0,8.6,29.9,3.1,...,17.71,20.70,3.85,10.51,50128.0,14.5,33.6968,21.0,32.6,6.0
8,1017,Chambers County,Alabama,6,13.2,41.2,10.5,8.7,31.5,3.4,...,19.54,40.00,2.52,10.48,43875.0,16.3,49.1480,21.7,36.9,6.9
9,1019,Cherokee County,Alabama,6,11.0,41.3,10.1,9.5,32.5,3.0,...,22.93,4.80,1.66,6.28,42509.0,14.7,85.7363,22.7,36.0,7.4


### q05 — Q5 - Native county measures + RUCC (ruralkg), state FIPS 14-28

2026-11-18T23:47:15+00:00 · `spoke-okn`, `ruralkg` · 936 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX sd: <https://purl.org/okn/frink/kg/spoke-okn/sdoh/>
PREFIX rst: <http://sail.ua.edu/ruralkg/settlementtype/>
# Q5 - Native county measures (spoke-okn: CHR 2023, AHRQ SDOH 2018-2020) pivoted one row per county + RUCC (ruralkg); state FIPS 14-28
SELECT ?fips ?county ?state ?rucc ?diabetes ?obesity ?asthma ?copd ?arthritis ?ckd ?high_chol ?mental_distress ?physical_distress ?heart_mort ?stroke_mort ?heart_hosp ?stroke_hosp ?pop ?age65 ?black ?hispanic ?bachelor ?income ?poverty ?rural ?smoking ?inactivity ?food_env WHERE {
 { SELECT ?fips (SAMPLE(?cn) AS ?county) (SAMPLE(?sn) AS ?state)
    (MAX(IF(?col="diabetes",?num,?u)) AS ?diabetes) (MAX(IF(?col="obesity",?num,?u)) AS ?obesity) (MAX(IF(?col="asthma",?num,?u)) AS ?asthma)
    (MAX(IF(?col="copd",?num,?u)) AS ?copd) (MAX(IF(?col="arthritis",?num,?u)) AS ?arthritis) (MAX(IF(?col="ckd",?num,?u)) AS ?ckd)
    (MAX(IF(?col="high_chol",?num,?u)) AS ?high_chol) (MAX(IF(?col="mental_distress",?num,?u)) AS ?mental_distress) (MAX(IF(?col="physical_distress",?num,?u)) AS ?physical_distress)
    (MAX(IF(?col="heart_mort",?num,?u)) AS ?heart_mort) (MAX(IF(?col="stroke_mort",?num,?u)) AS ?stroke_mort) (MAX(IF(?col="heart_hosp",?num,?u)) AS ?heart_hosp) (MAX(IF(?col="stroke_hosp",?num,?u)) AS ?stroke_hosp)
    (MAX(IF(?col="pop",?num,?u)) AS ?pop) (MAX(IF(?col="age65",?num,?u)) AS ?age65) (MAX(IF(?col="black",?num,?u)) AS ?black) (MAX(IF(?col="hispanic",?num,?u)) AS ?hispanic)
    (MAX(IF(?col="bachelor",?num,?u)) AS ?bachelor) (MAX(IF(?col="income",?num,?u)) AS ?income) (MAX(IF(?col="poverty",?num,?u)) AS ?poverty) (MAX(IF(?col="rural",?num,?u)) AS ?rural)
    (MAX(IF(?col="smoking",?num,?u)) AS ?smoking) (MAX(IF(?col="inactivity",?num,?u)) AS ?inactivity) (MAX(IF(?col="food_env",?num,?u)) AS ?food_env)
   WHERE {
    VALUES (?sdoh ?col ?yr) {
      (sd:CHR_diabetes_prevalence "diabetes" "2023") (sd:414916001 "obesity" "2023") (sd:CDCP_ASTHMA_ADULT_A "asthma" "2019")
      (sd:CDCP_PULMONARY_ADULT_A "copd" "2019") (sd:CDCP_ARTHRITIS_ADULT_A "arthritis" "2019") (sd:CDCP_KIDNEY_DISEASE_ADULT_A "ckd" "2019")
      (sd:CDCP_CHOLES_ADULT_A "high_chol" "2019") (sd:CHR_frequent_mental_distress "mental_distress" "2023") (sd:CHR_frequent_physical_distress "physical_distress" "2023")
      (sd:CDCA_HEART_DTH_RATE_ABOVE35 "heart_mort" "2019") (sd:CDCA_STROKE_DTH_RATE_ABOVE35 "stroke_mort" "2019")
      (sd:CDCA_HEART_HOSP_RATE_ABOVE65 "heart_hosp" "2018") (sd:CDCA_STROKE_HOSP_RATE_ABOVE65 "stroke_hosp" "2018")
      (sd:ACS_TOT_POP_WT "pop" "2020") (sd:ACS_PCT_AGE_ABOVE65 "age65" "2020") (sd:ACS_PCT_BLACK_NONHISP "black" "2020")
      (sd:ACS_PCT_HISPANIC "hispanic" "2020") (sd:ACS_PCT_BACHELOR_DGR "bachelor" "2020") (sd:ACS_MEDIAN_HH_INC "income" "2020")
      (sd:SAIPE_PCT_POV "poverty" "2020") (sd:CHR_rural "rural" "2023") (sd:77176002 "smoking" "2023")
      (sd:CDCP_NO_PHY_ACTV_ADULT_A "inactivity" "2019") (sd:CHR_food_environment_index "food_env" "2023") }
    GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?stmt rdf:subject ?sdoh ; rdf:predicate kg:PREVALENCEIN_SpL ; rdf:object ?loc ; kg:value ?val ; kg:year ?y .
      ?loc rdfs:label ?cn ; kg:state_name ?sn .
    }
    FILTER(STR(?y) = ?yr)
    BIND(STRAFTER(STR(?loc), "/location/") AS ?fips)
    FILTER(REGEX(?fips, "^(1[4-9]|2[0-8])[0-9]{3}$"))
    BIND(xsd:double(IF(CONTAINS(STR(?val),"("), STRBEFORE(STR(?val),"("), STR(?val))) AS ?num)
   } GROUP BY ?fips }
 BIND(IRI(CONCAT("http://stko-kwg.geog.ucsb.edu/lod/resource/administrativeRegion.USA.", ?fips)) AS ?reg)
 OPTIONAL { GRAPH <https://purl.org/okn/frink/kg/ruralkg> { ?cs rst:censusCounty ?reg ; rst:hasRUCC ?r . } }
 BIND(REPLACE(STR(?r), "^.*[^0-9]([0-9]+)$", "$1") AS ?rucc)
} ORDER BY ?fips LIMIT 999
```

In [6]:
data["q05"] = run_query("q05", rerun=RERUN)
data["q05"].head(10)

,fips,county,state,rucc,diabetes,obesity,asthma,copd,arthritis,ckd,...,age65,black,hispanic,bachelor,income,poverty,rural,smoking,inactivity,food_env
0,15001,Hawaii County,Hawaii,5,8.7,24.8,10.0,5.1,18.8,2.7,...,20.92,0.60,13.20,19.55,65401.0,12.2,37.990800,16.5,24.9,7.1
1,15003,Honolulu County,Hawaii,2,9.6,24.3,9.2,3.9,18.2,2.4,...,17.79,2.31,10.05,23.12,87722.0,8.0,0.862877,12.0,25.1,7.9
2,15007,Kauai County,Hawaii,5,8.3,22.0,9.2,4.3,20.0,2.5,...,20.05,0.45,11.37,20.24,82818.0,9.9,12.860100,13.4,25.1,7.2
3,15009,Maui County,Hawaii,3,8.5,23.9,8.8,4.4,18.4,2.5,...,18.25,0.59,11.63,18.78,84363.0,9.5,14.567900,14.0,24.2,7.1
4,16001,Ada County,Idaho,2,7.5,27.7,9.0,4.8,21.0,2.4,...,14.46,1.14,8.42,26.51,69952.0,7.7,5.472200,13.3,21.1,8.6
5,16003,Adams County,Idaho,9,8.8,32.8,9.9,6.1,23.3,2.6,...,30.14,0.33,3.76,11.90,50309.0,13.7,100.000000,17.5,24.4,7.2
6,16005,Bannock County,Idaho,3,9.2,36.3,9.7,5.9,24.7,2.8,...,14.32,0.57,8.88,18.38,51977.0,12.2,15.729300,18.3,26.8,7.5
7,16007,Bear Lake County,Idaho,9,9.2,35.1,10.0,6.4,25.0,2.8,...,20.60,0.30,4.31,13.41,52829.0,9.5,100.000000,17.8,25.3,8.2
8,16009,Benewah County,Idaho,6,9.9,33.5,10.5,7.3,25.9,3.0,...,22.59,0.39,3.95,12.26,47983.0,12.5,71.911700,19.7,26.4,7.3
9,16011,Bingham County,Idaho,4,10.3,39.3,9.8,5.9,23.2,2.9,...,13.96,0.19,18.17,14.21,58260.0,11.1,56.030900,17.5,26.1,7.9


### q06 — Q6 - Native county measures + RUCC (ruralkg), state FIPS 29-40

2026-11-18T23:47:38+00:00 · `spoke-okn`, `ruralkg` · 723 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX sd: <https://purl.org/okn/frink/kg/spoke-okn/sdoh/>
PREFIX rst: <http://sail.ua.edu/ruralkg/settlementtype/>
# Q6 - Native county measures (spoke-okn: CHR 2023, AHRQ SDOH 2018-2020) pivoted one row per county + RUCC (ruralkg); state FIPS 29-40
SELECT ?fips ?county ?state ?rucc ?diabetes ?obesity ?asthma ?copd ?arthritis ?ckd ?high_chol ?mental_distress ?physical_distress ?heart_mort ?stroke_mort ?heart_hosp ?stroke_hosp ?pop ?age65 ?black ?hispanic ?bachelor ?income ?poverty ?rural ?smoking ?inactivity ?food_env WHERE {
 { SELECT ?fips (SAMPLE(?cn) AS ?county) (SAMPLE(?sn) AS ?state)
    (MAX(IF(?col="diabetes",?num,?u)) AS ?diabetes) (MAX(IF(?col="obesity",?num,?u)) AS ?obesity) (MAX(IF(?col="asthma",?num,?u)) AS ?asthma)
    (MAX(IF(?col="copd",?num,?u)) AS ?copd) (MAX(IF(?col="arthritis",?num,?u)) AS ?arthritis) (MAX(IF(?col="ckd",?num,?u)) AS ?ckd)
    (MAX(IF(?col="high_chol",?num,?u)) AS ?high_chol) (MAX(IF(?col="mental_distress",?num,?u)) AS ?mental_distress) (MAX(IF(?col="physical_distress",?num,?u)) AS ?physical_distress)
    (MAX(IF(?col="heart_mort",?num,?u)) AS ?heart_mort) (MAX(IF(?col="stroke_mort",?num,?u)) AS ?stroke_mort) (MAX(IF(?col="heart_hosp",?num,?u)) AS ?heart_hosp) (MAX(IF(?col="stroke_hosp",?num,?u)) AS ?stroke_hosp)
    (MAX(IF(?col="pop",?num,?u)) AS ?pop) (MAX(IF(?col="age65",?num,?u)) AS ?age65) (MAX(IF(?col="black",?num,?u)) AS ?black) (MAX(IF(?col="hispanic",?num,?u)) AS ?hispanic)
    (MAX(IF(?col="bachelor",?num,?u)) AS ?bachelor) (MAX(IF(?col="income",?num,?u)) AS ?income) (MAX(IF(?col="poverty",?num,?u)) AS ?poverty) (MAX(IF(?col="rural",?num,?u)) AS ?rural)
    (MAX(IF(?col="smoking",?num,?u)) AS ?smoking) (MAX(IF(?col="inactivity",?num,?u)) AS ?inactivity) (MAX(IF(?col="food_env",?num,?u)) AS ?food_env)
   WHERE {
    VALUES (?sdoh ?col ?yr) {
      (sd:CHR_diabetes_prevalence "diabetes" "2023") (sd:414916001 "obesity" "2023") (sd:CDCP_ASTHMA_ADULT_A "asthma" "2019")
      (sd:CDCP_PULMONARY_ADULT_A "copd" "2019") (sd:CDCP_ARTHRITIS_ADULT_A "arthritis" "2019") (sd:CDCP_KIDNEY_DISEASE_ADULT_A "ckd" "2019")
      (sd:CDCP_CHOLES_ADULT_A "high_chol" "2019") (sd:CHR_frequent_mental_distress "mental_distress" "2023") (sd:CHR_frequent_physical_distress "physical_distress" "2023")
      (sd:CDCA_HEART_DTH_RATE_ABOVE35 "heart_mort" "2019") (sd:CDCA_STROKE_DTH_RATE_ABOVE35 "stroke_mort" "2019")
      (sd:CDCA_HEART_HOSP_RATE_ABOVE65 "heart_hosp" "2018") (sd:CDCA_STROKE_HOSP_RATE_ABOVE65 "stroke_hosp" "2018")
      (sd:ACS_TOT_POP_WT "pop" "2020") (sd:ACS_PCT_AGE_ABOVE65 "age65" "2020") (sd:ACS_PCT_BLACK_NONHISP "black" "2020")
      (sd:ACS_PCT_HISPANIC "hispanic" "2020") (sd:ACS_PCT_BACHELOR_DGR "bachelor" "2020") (sd:ACS_MEDIAN_HH_INC "income" "2020")
      (sd:SAIPE_PCT_POV "poverty" "2020") (sd:CHR_rural "rural" "2023") (sd:77176002 "smoking" "2023")
      (sd:CDCP_NO_PHY_ACTV_ADULT_A "inactivity" "2019") (sd:CHR_food_environment_index "food_env" "2023") }
    GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?stmt rdf:subject ?sdoh ; rdf:predicate kg:PREVALENCEIN_SpL ; rdf:object ?loc ; kg:value ?val ; kg:year ?y .
      ?loc rdfs:label ?cn ; kg:state_name ?sn .
    }
    FILTER(STR(?y) = ?yr)
    BIND(STRAFTER(STR(?loc), "/location/") AS ?fips)
    FILTER(REGEX(?fips, "^(29|3[0-9]|40)[0-9]{3}$"))
    BIND(xsd:double(IF(CONTAINS(STR(?val),"("), STRBEFORE(STR(?val),"("), STR(?val))) AS ?num)
   } GROUP BY ?fips }
 BIND(IRI(CONCAT("http://stko-kwg.geog.ucsb.edu/lod/resource/administrativeRegion.USA.", ?fips)) AS ?reg)
 OPTIONAL { GRAPH <https://purl.org/okn/frink/kg/ruralkg> { ?cs rst:censusCounty ?reg ; rst:hasRUCC ?r . } }
 BIND(REPLACE(STR(?r), "^.*[^0-9]([0-9]+)$", "$1") AS ?rucc)
} ORDER BY ?fips LIMIT 999
```

In [7]:
data["q06"] = run_query("q06", rerun=RERUN)
data["q06"].head(10)

,fips,county,state,rucc,diabetes,obesity,asthma,copd,arthritis,ckd,...,age65,black,hispanic,bachelor,income,poverty,rural,smoking,inactivity,food_env
0,29001,Adair County,Missouri,7,10.3,40.2,10.2,8.6,27.7,3.0,...,14.73,3.70,2.54,14.40,42301.0,17.9,37.8529,22.0,31.4,6.9
1,29003,Andrew County,Missouri,3,8.6,38.1,9.6,7.1,24.8,2.5,...,19.18,1.07,2.54,16.43,58911.0,9.1,61.2804,18.6,29.1,8.4
2,29005,Atchison County,Missouri,9,9.3,36.1,9.9,7.7,25.8,2.6,...,25.48,0.41,1.47,16.94,51625.0,12.0,100.0000,21.1,30.3,8.2
3,29007,Audrain County,Missouri,6,10.7,39.7,11.0,9.1,27.2,2.9,...,17.82,5.50,3.18,9.36,44699.0,17.8,41.1963,23.6,36.8,7.1
4,29009,Barry County,Missouri,6,10.8,38.4,10.2,9.2,26.7,3.0,...,21.19,0.22,9.38,10.24,45811.0,16.0,73.2646,23.7,37.6,7.0
5,29011,Barton County,Missouri,6,10.5,39.6,10.6,9.4,27.0,3.0,...,20.79,0.17,2.79,13.15,44510.0,16.6,63.9574,24.7,35.0,4.7
6,29013,Bates County,Missouri,1,10.2,41.2,10.3,8.7,26.7,2.7,...,19.54,1.83,2.40,10.13,46733.0,14.8,77.2714,24.0,33.3,7.2
7,29015,Benton County,Missouri,7,10.4,39.9,10.5,9.3,27.2,2.9,...,30.91,0.41,2.08,7.50,41751.0,16.8,86.5397,24.4,35.2,6.9
8,29017,Bollinger County,Missouri,3,10.6,39.1,10.7,10.0,28.0,3.0,...,20.01,0.56,1.40,6.71,45140.0,16.3,100.0000,26.0,38.4,6.8
9,29019,Boone County,Missouri,3,9.2,32.8,9.6,6.8,24.7,2.7,...,12.26,8.85,3.47,26.98,58740.0,12.8,18.7860,15.9,29.3,7.5


### q07 — Q7 - Native county measures + RUCC (ruralkg), state FIPS 41-56

2026-11-18T23:48:02+00:00 · `spoke-okn`, `ruralkg` · 928 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX sd: <https://purl.org/okn/frink/kg/spoke-okn/sdoh/>
PREFIX rst: <http://sail.ua.edu/ruralkg/settlementtype/>
# Q7 - Native county measures (spoke-okn: CHR 2023, AHRQ SDOH 2018-2020) pivoted one row per county + RUCC (ruralkg); state FIPS 41-56
SELECT ?fips ?county ?state ?rucc ?diabetes ?obesity ?asthma ?copd ?arthritis ?ckd ?high_chol ?mental_distress ?physical_distress ?heart_mort ?stroke_mort ?heart_hosp ?stroke_hosp ?pop ?age65 ?black ?hispanic ?bachelor ?income ?poverty ?rural ?smoking ?inactivity ?food_env WHERE {
 { SELECT ?fips (SAMPLE(?cn) AS ?county) (SAMPLE(?sn) AS ?state)
    (MAX(IF(?col="diabetes",?num,?u)) AS ?diabetes) (MAX(IF(?col="obesity",?num,?u)) AS ?obesity) (MAX(IF(?col="asthma",?num,?u)) AS ?asthma)
    (MAX(IF(?col="copd",?num,?u)) AS ?copd) (MAX(IF(?col="arthritis",?num,?u)) AS ?arthritis) (MAX(IF(?col="ckd",?num,?u)) AS ?ckd)
    (MAX(IF(?col="high_chol",?num,?u)) AS ?high_chol) (MAX(IF(?col="mental_distress",?num,?u)) AS ?mental_distress) (MAX(IF(?col="physical_distress",?num,?u)) AS ?physical_distress)
    (MAX(IF(?col="heart_mort",?num,?u)) AS ?heart_mort) (MAX(IF(?col="stroke_mort",?num,?u)) AS ?stroke_mort) (MAX(IF(?col="heart_hosp",?num,?u)) AS ?heart_hosp) (MAX(IF(?col="stroke_hosp",?num,?u)) AS ?stroke_hosp)
    (MAX(IF(?col="pop",?num,?u)) AS ?pop) (MAX(IF(?col="age65",?num,?u)) AS ?age65) (MAX(IF(?col="black",?num,?u)) AS ?black) (MAX(IF(?col="hispanic",?num,?u)) AS ?hispanic)
    (MAX(IF(?col="bachelor",?num,?u)) AS ?bachelor) (MAX(IF(?col="income",?num,?u)) AS ?income) (MAX(IF(?col="poverty",?num,?u)) AS ?poverty) (MAX(IF(?col="rural",?num,?u)) AS ?rural)
    (MAX(IF(?col="smoking",?num,?u)) AS ?smoking) (MAX(IF(?col="inactivity",?num,?u)) AS ?inactivity) (MAX(IF(?col="food_env",?num,?u)) AS ?food_env)
   WHERE {
    VALUES (?sdoh ?col ?yr) {
      (sd:CHR_diabetes_prevalence "diabetes" "2023") (sd:414916001 "obesity" "2023") (sd:CDCP_ASTHMA_ADULT_A "asthma" "2019")
      (sd:CDCP_PULMONARY_ADULT_A "copd" "2019") (sd:CDCP_ARTHRITIS_ADULT_A "arthritis" "2019") (sd:CDCP_KIDNEY_DISEASE_ADULT_A "ckd" "2019")
      (sd:CDCP_CHOLES_ADULT_A "high_chol" "2019") (sd:CHR_frequent_mental_distress "mental_distress" "2023") (sd:CHR_frequent_physical_distress "physical_distress" "2023")
      (sd:CDCA_HEART_DTH_RATE_ABOVE35 "heart_mort" "2019") (sd:CDCA_STROKE_DTH_RATE_ABOVE35 "stroke_mort" "2019")
      (sd:CDCA_HEART_HOSP_RATE_ABOVE65 "heart_hosp" "2018") (sd:CDCA_STROKE_HOSP_RATE_ABOVE65 "stroke_hosp" "2018")
      (sd:ACS_TOT_POP_WT "pop" "2020") (sd:ACS_PCT_AGE_ABOVE65 "age65" "2020") (sd:ACS_PCT_BLACK_NONHISP "black" "2020")
      (sd:ACS_PCT_HISPANIC "hispanic" "2020") (sd:ACS_PCT_BACHELOR_DGR "bachelor" "2020") (sd:ACS_MEDIAN_HH_INC "income" "2020")
      (sd:SAIPE_PCT_POV "poverty" "2020") (sd:CHR_rural "rural" "2023") (sd:77176002 "smoking" "2023")
      (sd:CDCP_NO_PHY_ACTV_ADULT_A "inactivity" "2019") (sd:CHR_food_environment_index "food_env" "2023") }
    GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?stmt rdf:subject ?sdoh ; rdf:predicate kg:PREVALENCEIN_SpL ; rdf:object ?loc ; kg:value ?val ; kg:year ?y .
      ?loc rdfs:label ?cn ; kg:state_name ?sn .
    }
    FILTER(STR(?y) = ?yr)
    BIND(STRAFTER(STR(?loc), "/location/") AS ?fips)
    FILTER(REGEX(?fips, "^(4[1-9]|5[0-6])[0-9]{3}$"))
    BIND(xsd:double(IF(CONTAINS(STR(?val),"("), STRBEFORE(STR(?val),"("), STR(?val))) AS ?num)
   } GROUP BY ?fips }
 BIND(IRI(CONCAT("http://stko-kwg.geog.ucsb.edu/lod/resource/administrativeRegion.USA.", ?fips)) AS ?reg)
 OPTIONAL { GRAPH <https://purl.org/okn/frink/kg/ruralkg> { ?cs rst:censusCounty ?reg ; rst:hasRUCC ?r . } }
 BIND(REPLACE(STR(?r), "^.*[^0-9]([0-9]+)$", "$1") AS ?rucc)
} ORDER BY ?fips LIMIT 999
```

In [8]:
data["q07"] = run_query("q07", rerun=RERUN)
data["q07"].head(10)

,fips,county,state,rucc,diabetes,obesity,asthma,copd,arthritis,ckd,...,age65,black,hispanic,bachelor,income,poverty,rural,smoking,inactivity,food_env
0,41001,Baker County,Oregon,7,8.6,28.0,10.5,6.4,25.2,2.8,...,26.68,1.04,4.57,17.63,46250.0,14.5,41.0066,17.5,25.7,6.7
1,41003,Benton County,Oregon,3,8.2,28.5,10.3,5.2,23.3,2.6,...,16.03,1.06,7.63,29.23,65142.0,15.7,18.7639,12.7,22.2,8.0
2,41005,Clackamas County,Oregon,1,7.6,25.8,9.5,4.6,22.5,2.4,...,18.05,0.82,8.88,24.57,82911.0,6.8,18.0786,13.3,22.3,8.8
3,41007,Clatsop County,Oregon,4,8.4,28.5,10.5,6.0,23.9,2.7,...,22.27,0.75,8.68,15.45,57466.0,10.5,38.9724,16.9,26.0,7.9
4,41009,Columbia County,Oregon,1,8.2,30.2,10.5,6.3,24.9,2.7,...,18.69,0.49,5.44,12.04,68170.0,9.2,43.6121,16.7,27.1,8.6
5,41011,Coos County,Oregon,5,9.2,34.0,11.4,7.1,25.6,3.0,...,25.87,0.36,6.70,12.47,49445.0,15.4,38.3532,19.1,27.8,7.5
6,41013,Crook County,Oregon,6,8.5,29.8,10.7,6.5,25.1,2.8,...,24.88,0.14,7.70,13.40,59000.0,10.3,48.0170,18.1,26.9,7.9
7,41015,Curry County,Oregon,7,8.6,28.9,10.5,6.2,23.9,2.7,...,34.35,0.15,7.35,13.63,53174.0,14.5,38.7319,17.3,26.7,8.0
8,41017,Deschutes County,Oregon,3,7.8,27.5,9.7,5.2,21.0,2.5,...,19.89,0.52,8.09,22.83,68937.0,8.1,27.6435,14.0,23.5,8.3
9,41019,Douglas County,Oregon,4,8.6,31.0,10.9,6.9,24.6,2.9,...,25.66,0.55,5.93,12.03,50031.0,13.3,41.1779,18.6,27.9,7.3


### q08 — Q8 - PLACES place-level prevalence rolled up to counties, state FIPS 01-13

2026-11-18T23:48:19+00:00 · `spoke-okn` · 476 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX doid: <http://purl.obolibrary.org/obo/DOID_>
# Q8 - CDC PLACES place-level age-adjusted prevalence (spoke-okn PREVALENCE_DpL on City nodes) rolled up to counties via PARTOF_LpL,
#      population-weighted (place population split evenly across the counties a place lies in); state FIPS 01-13
SELECT ?fips (COUNT(DISTINCT ?city) AS ?n_places) (ROUND(SUM(IF(?col="htn",?w,0))) AS ?pop_places)
  (ROUND(10*SUM(IF(?col="htn",?dv*?w,0))/SUM(IF(?col="htn",?w,0)))/10 AS ?hypertension)
  (ROUND(10*SUM(IF(?col="chd",?dv*?w,0))/SUM(IF(?col="chd",?w,0)))/10 AS ?chd)
  (ROUND(10*SUM(IF(?col="stroke",?dv*?w,0))/SUM(IF(?col="stroke",?w,0)))/10 AS ?stroke)
  (ROUND(10*SUM(IF(?col="dep",?dv*?w,0))/SUM(IF(?col="dep",?w,0)))/10 AS ?depression)
  (ROUND(10*SUM(IF(?col="dm",?dv*?w,0))/SUM(IF(?col="dm",?w,0)))/10 AS ?diabetes_pl)
  (ROUND(10*SUM(IF(?col="ob",?dv*?w,0))/SUM(IF(?col="ob",?w,0)))/10 AS ?obesity_pl)
  (ROUND(10*SUM(IF(?col="ast",?dv*?w,0))/SUM(IF(?col="ast",?w,0)))/10 AS ?asthma_pl)
  (ROUND(10*SUM(IF(?col="copd",?dv*?w,0))/SUM(IF(?col="copd",?w,0)))/10 AS ?copd_pl)
  (ROUND(10*SUM(IF(?col="chol",?dv*?w,0))/SUM(IF(?col="chol",?w,0)))/10 AS ?high_chol_pl)
WHERE {
  { SELECT ?city (COUNT(DISTINCT ?c2) AS ?nc) WHERE { GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?city kg:PARTOF_LpL ?c2 . ?c2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> . } } GROUP BY ?city }
  VALUES (?d ?col) { (doid:10763 "htn") (doid:3393 "chd") (doid:6713 "stroke") (doid:1596 "dep") (doid:9351 "dm") (doid:9970 "ob") (doid:2841 "ast") (doid:3083 "copd") (doid:2349 "chol") }
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:subject ?d ; rdf:predicate kg:PREVALENCE_DpL ; rdf:object ?city ; kg:data_value ?dvs ; kg:total_population ?tp .
    ?city kg:PARTOF_LpL ?cty . ?cty <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> .
  }
  BIND(STRAFTER(STR(?cty), "/location/") AS ?fips)
  FILTER(REGEX(?fips, "^(0[1-9]|1[0-3])[0-9]{3}$"))
  BIND(xsd:double(?dvs) AS ?dv)
  BIND(xsd:double(?tp) / ?nc AS ?w)
} GROUP BY ?fips ORDER BY ?fips
```

In [9]:
data["q08"] = run_query("q08", rerun=RERUN)
data["q08"].head(10)

,fips,n_places,pop_places,hypertension,chd,stroke,depression,diabetes_pl,obesity_pl,asthma_pl,copd_pl,high_chol_pl
0,1001,4,6615.0,36.4,5.8,3.0,23.5,10.3,38.2,10.1,7.1,32.4
1,1003,15,98544.0,32.7,5.4,2.7,22.9,9.8,37.2,9.9,6.4,31.8
2,1005,6,18438.0,44.1,7.6,4.5,23.9,15.9,43.6,11.9,10.4,33.7
3,1007,3,8965.0,41.9,6.9,3.9,21.9,13.5,41.2,10.5,8.9,34.0
4,1009,12,16979.0,35.7,6.4,3.2,24.6,10.9,38.3,10.4,8.3,34.3
5,1011,3,4562.0,48.7,7.5,5.2,20.9,18.9,49.6,12.6,9.9,32.8
6,1013,2,9873.0,44.1,6.7,4.2,21.3,15.5,43.8,12.0,8.9,33.5
7,1015,11,63128.0,40.1,6.5,3.6,23.3,13.0,40.5,11.2,8.6,33.2
8,1017,10,22593.0,41.2,6.3,3.8,22.9,13.6,44.5,11.2,8.2,32.7
9,1019,6,6914.0,37.0,6.5,3.3,26.4,11.2,38.2,10.8,8.8,33.3


### q09 — Q9 - PLACES place-level prevalence rolled up to counties, state FIPS 14-28

2026-11-18T23:48:41+00:00 · `spoke-okn` · 936 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX doid: <http://purl.obolibrary.org/obo/DOID_>
# Q9 - CDC PLACES place-level age-adjusted prevalence rolled up to counties (population-weighted, see Q8); state FIPS 14-28
SELECT ?fips (COUNT(DISTINCT ?city) AS ?n_places) (ROUND(SUM(IF(?col="htn",?w,0))) AS ?pop_places)
  (ROUND(10*SUM(IF(?col="htn",?dv*?w,0))/SUM(IF(?col="htn",?w,0)))/10 AS ?hypertension)
  (ROUND(10*SUM(IF(?col="chd",?dv*?w,0))/SUM(IF(?col="chd",?w,0)))/10 AS ?chd)
  (ROUND(10*SUM(IF(?col="stroke",?dv*?w,0))/SUM(IF(?col="stroke",?w,0)))/10 AS ?stroke)
  (ROUND(10*SUM(IF(?col="dep",?dv*?w,0))/SUM(IF(?col="dep",?w,0)))/10 AS ?depression)
  (ROUND(10*SUM(IF(?col="dm",?dv*?w,0))/SUM(IF(?col="dm",?w,0)))/10 AS ?diabetes_pl)
  (ROUND(10*SUM(IF(?col="ob",?dv*?w,0))/SUM(IF(?col="ob",?w,0)))/10 AS ?obesity_pl)
  (ROUND(10*SUM(IF(?col="ast",?dv*?w,0))/SUM(IF(?col="ast",?w,0)))/10 AS ?asthma_pl)
  (ROUND(10*SUM(IF(?col="copd",?dv*?w,0))/SUM(IF(?col="copd",?w,0)))/10 AS ?copd_pl)
  (ROUND(10*SUM(IF(?col="chol",?dv*?w,0))/SUM(IF(?col="chol",?w,0)))/10 AS ?high_chol_pl)
WHERE {
  { SELECT ?city (COUNT(DISTINCT ?c2) AS ?nc) WHERE { GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?city kg:PARTOF_LpL ?c2 . ?c2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> . } } GROUP BY ?city }
  VALUES (?d ?col) { (doid:10763 "htn") (doid:3393 "chd") (doid:6713 "stroke") (doid:1596 "dep") (doid:9351 "dm") (doid:9970 "ob") (doid:2841 "ast") (doid:3083 "copd") (doid:2349 "chol") }
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:subject ?d ; rdf:predicate kg:PREVALENCE_DpL ; rdf:object ?city ; kg:data_value ?dvs ; kg:total_population ?tp .
    ?city kg:PARTOF_LpL ?cty . ?cty <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> .
  }
  BIND(STRAFTER(STR(?cty), "/location/") AS ?fips)
  FILTER(REGEX(?fips, "^(1[4-9]|2[0-8])[0-9]{3}$"))
  BIND(xsd:double(?dvs) AS ?dv)
  BIND(xsd:double(?tp) / ?nc AS ?w)
} GROUP BY ?fips ORDER BY ?fips
```

In [10]:
data["q09"] = run_query("q09", rerun=RERUN)
data["q09"].head(10)

,fips,n_places,pop_places,hypertension,chd,stroke,depression,diabetes_pl,obesity_pl,asthma_pl,copd_pl,high_chol_pl
0,15001,44,165415.0,25.3,4.8,2.8,17.2,8.8,26.6,10.0,5.0,26.8
1,15003,51,928797.0,26.4,3.8,2.5,12.2,9.2,24.4,8.4,3.6,29.8
2,15007,24,62979.0,24.6,4.4,2.6,14.5,8.3,23.2,9.0,4.3,26.4
3,15009,28,148823.0,22.2,4.4,2.6,13.3,8.0,23.8,9.1,4.2,25.9
4,16001,6,335567.0,26.2,4.9,2.5,25.8,7.9,28.8,9.8,5.0,26.7
5,16003,2,1335.0,28.8,5.6,2.8,23.7,8.7,30.6,10.4,6.3,27.8
6,16005,6,17053.0,29.6,5.3,2.8,24.0,8.9,37.2,10.5,6.3,29.1
7,16007,6,4113.0,30.8,5.7,2.9,23.7,9.4,34.9,10.7,7.0,28.1
8,16009,4,3748.0,30.9,6.2,3.3,24.4,10.3,33.9,11.5,7.6,27.4
9,16011,9,22442.0,28.8,5.4,2.8,23.9,9.4,36.2,10.8,6.4,26.8


### q10 — Q10 - PLACES place-level prevalence rolled up to counties, state FIPS 29-40

2026-11-18T23:48:56+00:00 · `spoke-okn` · 718 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX doid: <http://purl.obolibrary.org/obo/DOID_>
# Q10 - CDC PLACES place-level age-adjusted prevalence rolled up to counties (population-weighted, see Q8); state FIPS 29-40
SELECT ?fips (COUNT(DISTINCT ?city) AS ?n_places) (ROUND(SUM(IF(?col="htn",?w,0))) AS ?pop_places)
  (ROUND(10*SUM(IF(?col="htn",?dv*?w,0))/SUM(IF(?col="htn",?w,0)))/10 AS ?hypertension)
  (ROUND(10*SUM(IF(?col="chd",?dv*?w,0))/SUM(IF(?col="chd",?w,0)))/10 AS ?chd)
  (ROUND(10*SUM(IF(?col="stroke",?dv*?w,0))/SUM(IF(?col="stroke",?w,0)))/10 AS ?stroke)
  (ROUND(10*SUM(IF(?col="dep",?dv*?w,0))/SUM(IF(?col="dep",?w,0)))/10 AS ?depression)
  (ROUND(10*SUM(IF(?col="dm",?dv*?w,0))/SUM(IF(?col="dm",?w,0)))/10 AS ?diabetes_pl)
  (ROUND(10*SUM(IF(?col="ob",?dv*?w,0))/SUM(IF(?col="ob",?w,0)))/10 AS ?obesity_pl)
  (ROUND(10*SUM(IF(?col="ast",?dv*?w,0))/SUM(IF(?col="ast",?w,0)))/10 AS ?asthma_pl)
  (ROUND(10*SUM(IF(?col="copd",?dv*?w,0))/SUM(IF(?col="copd",?w,0)))/10 AS ?copd_pl)
  (ROUND(10*SUM(IF(?col="chol",?dv*?w,0))/SUM(IF(?col="chol",?w,0)))/10 AS ?high_chol_pl)
WHERE {
  { SELECT ?city (COUNT(DISTINCT ?c2) AS ?nc) WHERE { GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?city kg:PARTOF_LpL ?c2 . ?c2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> . } } GROUP BY ?city }
  VALUES (?d ?col) { (doid:10763 "htn") (doid:3393 "chd") (doid:6713 "stroke") (doid:1596 "dep") (doid:9351 "dm") (doid:9970 "ob") (doid:2841 "ast") (doid:3083 "copd") (doid:2349 "chol") }
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:subject ?d ; rdf:predicate kg:PREVALENCE_DpL ; rdf:object ?city ; kg:data_value ?dvs ; kg:total_population ?tp .
    ?city kg:PARTOF_LpL ?cty . ?cty <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> .
  }
  BIND(STRAFTER(STR(?cty), "/location/") AS ?fips)
  FILTER(REGEX(?fips, "^(29|3[0-9]|40)[0-9]{3}$"))
  BIND(xsd:double(?dvs) AS ?dv)
  BIND(xsd:double(?tp) / ?nc AS ?w)
} GROUP BY ?fips ORDER BY ?fips
```

In [11]:
data["q10"] = run_query("q10", rerun=RERUN)
data["q10"].head(10)

,fips,n_places,pop_places,hypertension,chd,stroke,depression,diabetes_pl,obesity_pl,asthma_pl,copd_pl,high_chol_pl
0,29001,4,18323.0,32.6,6.5,3.2,27.6,10.6,38.0,10.6,9.0,32.5
1,29003,7,8456.0,30.9,5.4,2.7,24.1,8.8,38.1,9.6,7.1,30.7
2,29005,4,3680.0,31.8,5.7,2.8,25.3,9.1,37.7,9.6,7.5,30.5
3,29007,7,13005.0,35.3,6.2,3.3,25.6,11.4,42.0,10.4,8.8,32.8
4,29009,12,8558.0,34.0,6.9,3.5,27.0,11.4,41.3,10.8,10.2,32.2
5,29011,6,6673.0,34.3,6.8,3.4,27.1,11.5,43.0,10.6,9.9,32.5
6,29013,8,8284.0,33.3,6.4,3.2,26.7,10.7,41.3,10.5,9.2,33.1
7,29015,3,4438.0,33.4,6.6,3.3,24.9,10.8,43.7,10.6,9.6,33.6
8,29017,4,1835.0,34.8,7.1,3.6,28.3,11.5,42.5,11.0,10.7,33.7
9,29019,8,115935.0,30.2,5.5,2.8,25.4,9.9,33.4,9.9,6.6,32.2


### q11 — Q11 - PLACES place-level prevalence rolled up to counties, state FIPS 41-56

2026-11-18T23:49:11+00:00 · `spoke-okn` · 927 rows logged

```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX kg: <https://purl.org/okn/frink/kg/spoke-okn/schema/>
PREFIX doid: <http://purl.obolibrary.org/obo/DOID_>
# Q11 - CDC PLACES place-level age-adjusted prevalence rolled up to counties (population-weighted, see Q8); state FIPS 41-56
SELECT ?fips (COUNT(DISTINCT ?city) AS ?n_places) (ROUND(SUM(IF(?col="htn",?w,0))) AS ?pop_places)
  (ROUND(10*SUM(IF(?col="htn",?dv*?w,0))/SUM(IF(?col="htn",?w,0)))/10 AS ?hypertension)
  (ROUND(10*SUM(IF(?col="chd",?dv*?w,0))/SUM(IF(?col="chd",?w,0)))/10 AS ?chd)
  (ROUND(10*SUM(IF(?col="stroke",?dv*?w,0))/SUM(IF(?col="stroke",?w,0)))/10 AS ?stroke)
  (ROUND(10*SUM(IF(?col="dep",?dv*?w,0))/SUM(IF(?col="dep",?w,0)))/10 AS ?depression)
  (ROUND(10*SUM(IF(?col="dm",?dv*?w,0))/SUM(IF(?col="dm",?w,0)))/10 AS ?diabetes_pl)
  (ROUND(10*SUM(IF(?col="ob",?dv*?w,0))/SUM(IF(?col="ob",?w,0)))/10 AS ?obesity_pl)
  (ROUND(10*SUM(IF(?col="ast",?dv*?w,0))/SUM(IF(?col="ast",?w,0)))/10 AS ?asthma_pl)
  (ROUND(10*SUM(IF(?col="copd",?dv*?w,0))/SUM(IF(?col="copd",?w,0)))/10 AS ?copd_pl)
  (ROUND(10*SUM(IF(?col="chol",?dv*?w,0))/SUM(IF(?col="chol",?w,0)))/10 AS ?high_chol_pl)
WHERE {
  { SELECT ?city (COUNT(DISTINCT ?c2) AS ?nc) WHERE { GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
      ?city kg:PARTOF_LpL ?c2 . ?c2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> . } } GROUP BY ?city }
  VALUES (?d ?col) { (doid:10763 "htn") (doid:3393 "chd") (doid:6713 "stroke") (doid:1596 "dep") (doid:9351 "dm") (doid:9970 "ob") (doid:2841 "ast") (doid:3083 "copd") (doid:2349 "chol") }
  GRAPH <https://purl.org/okn/frink/kg/spoke-okn> {
    ?stmt rdf:subject ?d ; rdf:predicate kg:PREVALENCE_DpL ; rdf:object ?city ; kg:data_value ?dvs ; kg:total_population ?tp .
    ?city kg:PARTOF_LpL ?cty . ?cty <http://www.w3.org/1999/02/22-rdf-syntax-ns#> <http://www.wikidata.org/entity/Q28575> .
  }
  BIND(STRAFTER(STR(?cty), "/location/") AS ?fips)
  FILTER(REGEX(?fips, "^(4[1-9]|5[0-6])[0-9]{3}$"))
  BIND(xsd:double(?dvs) AS ?dv)
  BIND(xsd:double(?tp) / ?nc AS ?w)
} GROUP BY ?fips ORDER BY ?fips
```

In [12]:
data["q11"] = run_query("q11", rerun=RERUN)
data["q11"].head(10)

,fips,n_places,pop_places,hypertension,chd,stroke,depression,diabetes_pl,obesity_pl,asthma_pl,copd_pl,high_chol_pl
0,41001,6,11353.0,28.9,5.7,2.9,27.5,8.7,30.6,11.2,7.0,27.8
1,41003,9,61079.0,26.4,5.1,2.5,24.6,8.1,28.0,10.7,5.7,26.8
2,41005,19,184491.0,26.2,4.7,2.4,26.0,7.3,32.1,10.9,5.3,25.9
3,41007,6,24410.0,27.4,5.4,2.7,25.8,8.4,29.9,11.1,6.4,25.4
4,41009,9,29944.0,27.2,5.5,2.8,27.4,8.6,32.7,11.3,6.8,26.3
5,41011,10,41547.0,30.5,6.0,3.1,28.8,9.5,36.1,11.8,7.8,25.9
6,41013,1,9276.0,28.1,5.6,2.9,26.8,8.9,32.9,11.4,7.0,28.4
7,41015,6,12870.0,27.8,5.4,2.8,26.9,8.6,31.9,11.3,6.7,26.6
8,41017,11,119872.0,25.3,5.0,2.5,24.2,7.3,25.9,10.8,5.6,27.4
9,41019,23,71925.0,29.5,5.9,3.0,27.1,9.4,34.3,11.5,7.4,27.5


## Analysis

# Analysis

**Question.** Which US counties carry the highest prevalence of common chronic diseases, singly and in combination, and how do those conditions co-vary once demographic and geographic confounders are considered?

**Unit of analysis:** county (n = 3,131 counties and county-equivalents in the 50 states + DC). **Level of inference:** ecological (area-level), based on *model-based small-area estimates*, not individual diagnoses.

All analysis code lives in `analysis.py` (figure styling in `okn_figstyle.py`); every number below is computed from the query results `data["q01"]` ... `data["q11"]` loaded above.

## 1. Data discovery: what the OKN federation offers at county level

`list_kgs` / `get_schema` were used to screen the federation. Graphs with place-based health payloads were checked:

| KG | What it holds for this question | Used |
|---|---|---|
| **spoke-okn** (v0.0.6, loaded 2026-03-16) | `PREVALENCE_DpL` (disease -> location): CDC **PLACES** age-adjusted prevalence for 9 conditions on 27,365 **places** (cities/towns/CDPs), and IHME 2019 for 144 conditions at **country** level only (q01, q02). `PREVALENCEIN_SpL` (SDoH -> location): **county-level** health measures from County Health Rankings 2023 and the AHRQ SDOH database (CDC PLACES county estimates 2019, CDC Heart & Stroke Atlas), plus ACS/SAIPE covariates (q03). `PARTOF_LpL` links places to counties. | yes |
| **ruralkg** (v0.2.10) | USDA Rural-Urban Continuum Codes for 3,234 counties, joined on county FIPS (verified recipe K2-county-fips) | yes (urbanicity confounder) |
| biohealth, nikg, dreamkg, others | no county-level chronic-disease prevalence in their schemas | no |

Two complementary routes therefore give county estimates:

* **Tier A - native county estimates** (q04-q07): diabetes and obesity (CHR 2023), asthma, COPD, arthritis, chronic kidney disease and high cholesterol (AHRQ SDOH 2019, CDC PLACES county model). 
* **Tier B - place-level PLACES rolled up to the county** (q08-q11): hypertension, coronary heart disease, stroke and depression are *only* available at place level in spoke-okn. Each place's estimate is weighted by its population (split evenly when a place spans several counties) and averaged within the county. Diabetes, obesity, asthma, COPD and high cholesterol are rolled up too, purely to validate the method against Tier A (Section 2).

CVD mortality / hospitalization rates and frequent mental/physical distress are carried as *related outcomes* for the correlation analysis but are not part of the composite.

In [13]:
import os, warnings
import numpy as np
import pandas as pd
from IPython.display import IFrame, display

import analysis as A
from okn_figstyle import apply_style

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
os.makedirs("figures", exist_ok=True); os.makedirs("outputs", exist_ok=True)
apply_style()

df = A.load_county_table(data)
print(f"{len(df):,} counties; {df['state_fips'].nunique()} states incl. DC; columns: {df.shape[1]}")
# measure catalogue: source, data year as recorded in the KG, KG node
catalogue = A.national_summary(df)
catalogue

3,131 counties; 51 states incl. DC; columns: 43


,condition,key,tier,source,year,kg_node,n_counties,pop_weighted_mean_%,county_median_%,p10,p90,min,max
0,Diabetes,diabetes,A,County Health Rankings (CHR),2023 release,sdoh/CHR_diabetes_prevalence,3129,10.05,10.1,8.0,13.30,5.6,21.5
1,Obesity,obesity,A,County Health Rankings (CHR),2023 release,sdoh/414916001 (SNOMED Obesity),3129,32.54,36.7,30.1,41.30,17.6,53.2
2,Asthma,asthma,A,AHRQ SDOH (CDC PLACES county),2019,sdoh/CDCP_ASTHMA_ADULT_A,3109,9.22,9.8,8.4,10.90,7.2,13.3
3,COPD,copd,A,AHRQ SDOH (CDC PLACES county),2019,sdoh/CDCP_PULMONARY_ADULT_A,3109,6.15,7.2,5.3,9.90,3.2,15.5
4,Arthritis,arthritis,A,AHRQ SDOH (CDC PLACES county),2019,sdoh/CDCP_ARTHRITIS_ADULT_A,3109,22.53,24.8,21.0,29.40,15.2,38.7
5,Chronic kidney disease,ckd,A,AHRQ SDOH (CDC PLACES county),2019,sdoh/CDCP_KIDNEY_DISEASE_ADULT_A,3109,2.85,2.9,2.4,3.50,2.0,5.9
6,High cholesterol,high_chol,A,AHRQ SDOH (CDC PLACES county),2019,sdoh/CDCP_CHOLES_ADULT_A,3109,28.88,29.6,26.3,32.60,22.3,36.9
7,Hypertension,hypertension,B,"CDC PLACES place-level, rolled up to county",not recorded,DOID:10763,2962,30.72,32.2,27.3,40.00,21.4,54.4
8,Coronary heart disease,chd,B,"CDC PLACES place-level, rolled up to county",not recorded,DOID:3393,2962,5.37,5.9,4.9,7.09,3.5,10.9
9,Stroke,stroke,B,"CDC PLACES place-level, rolled up to county",not recorded,DOID:6713,2962,2.94,3.1,2.5,4.00,1.8,6.6


## 2. Validation and data gaps

Checks: duplicate FIPS, value ranges (all prevalences must lie in 0-100%), missingness per measure and per state, and a **method check for the Tier-B roll-up**: for the five conditions available both natively and as place roll-ups, the two estimates are compared county by county (correlation, mean difference, Bland-Altman 95% limits of agreement), overall and by place coverage.

In [14]:
qc, qc_summary = A.qc_report(df)
print(qc_summary)
display(qc)
gaps = A.data_gaps(df)
display(gaps)

{'counties': 3131, 'states_incl_DC': 51, 'duplicate_fips': 0}


,measure,label,n_counties,missing,min,median,max,out_of_range_0_100
0,diabetes,Diabetes,3129,2,5.600000,10.100000,21.500000,0.0
1,obesity,Obesity,3129,2,17.600000,36.700000,53.200000,0.0
2,asthma,Asthma,3110,21,7.200000,9.800000,13.300000,0.0
3,copd,COPD,3110,21,3.200000,7.200000,15.500000,0.0
4,arthritis,Arthritis,3110,21,15.200000,24.800000,38.700000,0.0
5,ckd,Chronic kidney disease,3110,21,2.000000,2.900000,5.900000,0.0
6,high_chol,High cholesterol,3110,21,22.300000,29.600000,36.900000,0.0
7,hypertension,Hypertension,2962,169,21.400000,32.200000,54.400000,0.0
8,chd,Coronary heart disease,2962,169,3.500000,5.900000,10.900000,0.0
9,stroke,Stroke,2962,169,1.800000,3.100000,6.600000,0.0


,measure,counties_missing,states_entirely_missing,states_partly_missing (n)
0,Diabetes,2,-,"Alaska (1), Texas (1)"
1,Obesity,2,-,"Alaska (1), Texas (1)"
2,Asthma,21,New Jersey,-
3,COPD,21,New Jersey,-
4,Arthritis,21,New Jersey,-
5,Chronic kidney disease,21,New Jersey,-
6,High cholesterol,21,New Jersey,-
7,Hypertension,169,Florida,"Virginia (22), Georgia (16), Kentucky (10), Te..."
8,Coronary heart disease,169,Florida,"Virginia (22), Georgia (16), Kentucky (10), Te..."
9,Stroke,169,Florida,"Virginia (22), Georgia (16), Kentucky (10), Te..."


In [15]:
validation = A.validate_place_vs_native(df)
display(validation)
A.plot_validation(df, "figures/fig1_validation.png", 1);

,measure,subset,n,pearson_r,spearman_rho,mean_diff_place_minus_native,limits_of_agreement_95
0,Diabetes,all counties,3052,0.936,0.932,0.35,-1.4 to 2.1
1,Diabetes,coverage < 25%,476,0.845,0.827,0.21,-2.0 to 2.5
2,Diabetes,coverage >= 50%,1511,0.960,0.946,0.28,-1.1 to 1.7
3,Obesity,all counties,3052,0.827,0.770,1.69,-3.7 to 7.1
4,Obesity,coverage < 25%,476,0.774,0.710,1.37,-4.3 to 7.0
5,Obesity,coverage >= 50%,1511,0.860,0.806,1.70,-3.5 to 6.9
6,Asthma,all counties,3032,0.852,0.847,0.94,-0.1 to 2.0
7,Asthma,coverage < 25%,474,0.794,0.789,0.93,-0.3 to 2.1
8,Asthma,coverage >= 50%,1496,0.877,0.871,0.95,-0.0 to 1.9
9,COPD,all counties,3032,0.944,0.943,0.22,-1.0 to 1.4


[okn_figstyle] saved Figure 1: figures/fig1_validation.png


**Figure 1.** (A-E) Place roll-up vs. native county estimate for the five conditions available both ways (spoke-okn `PREVALENCE_DpL` rolled up through `PARTOF_LpL` vs. `PREVALENCEIN_SpL`); colour = share of the county population living in PLACES places; dashed line = identity. (F) Distribution of place coverage; dashed line = the 10% threshold below which Tier-B values are discarded.

*Interpretation:* the roll-up reproduces the native county ranking well (r = 0.77-0.94), even where places cover < 25% of the county, so the Tier-B estimates for hypertension, CHD, stroke and depression are usable for ranking. The roll-up runs systematically higher for obesity (+1.7 points) and high cholesterol (+1.8) - consistent with a later PLACES release and with age-adjusted vs. county estimates - so Tier-A and Tier-B values should be compared as ranks/z-scores, not as absolute percentages.

**Systematic gaps** (table above): *New Jersey* has no AHRQ/PLACES 2019 county values (asthma, COPD, arthritis, CKD, high cholesterol); *Florida* has no PLACES place-level rows at all, so hypertension, CHD, stroke and depression are missing for all 67 Florida counties; 22 Virginia independent cities and a few other counties lack a place-to-county link.

## 3. Composite disease-burden score

* Each of the 11 conditions is standardized across counties (z-score; county = unit, unweighted).
* **`cbs11`** = mean z over all 11 conditions (equal weights; requires all 11, n = 2,940 counties).
* Sensitivity variants: `cbs7` (7 native Tier-A conditions only), `cbs_avail` (mean over >= 6 available conditions - keeps NJ and FL on the map), `cbs_rank` (mean national percentile rank), `cbs_pc1` (first principal component) and `cbs_disease_only` (drops the risk-factor measures obesity, high cholesterol, hypertension).
* **Multimorbidity:** `n_top_quintile` = number of the 11 conditions on which the county falls in the national top 20%.

In [16]:
df = A.build_scores(df)
print("counties scored (cbs11):", int(df.cbs11.notna().sum()), "| cbs7:", int(df.cbs7.notna().sum()), "| cbs_avail:", int(df.cbs_avail.notna().sum()))
print(f"PC1 explains {A.SCORE_INFO['pc1_var_explained']:.1%} of variance; loadings:")
print(A.SCORE_INFO["pc1_loadings"].to_string())
agreement = A.score_agreement(df)
agreement

counties scored (cbs11): 2940 | cbs7: 3108 | cbs_avail: 3130
PC1 explains 66.1% of variance; loadings:
diabetes        0.323
obesity         0.288
asthma          0.263
copd            0.330
arthritis       0.283
ckd             0.317
high_chol       0.282
hypertension    0.338
chd             0.345
stroke          0.335
depression      0.173


,alternative,spearman_vs_cbs11,top100_overlap
0,cbs7,0.992,88
1,cbs_avail,1.000,100
2,cbs_rank,0.997,72
3,cbs_pc1,0.999,91
4,cbs_disease_only,0.984,89


In [17]:
top_all = A.top_counties(df, "cbs11", 25)
top_big = A.top_counties(df, "cbs11", 25, min_pop=50_000)
print("Top 25 counties, composite burden (all):"); display(top_all)
print("Top 25 counties with population >= 50,000:"); display(top_big)
A.plot_top_counties(df, "figures/fig2_top_counties.png", 2);

Top 25 counties, composite burden (all):


,rank,fips,county,state,pop,rucc,cbs11,n_top_quintile
0,1,22035,East Carroll Parish,Louisiana,6947.0,7,2.958638,10.0
1,2,28051,Holmes County,Mississippi,17414.0,6,2.876288,10.0
2,3,46121,Todd County,South Dakota,10308.0,9,2.839371,9.0
3,4,54047,McDowell County,West Virginia,18083.0,7,2.838566,11.0
4,5,01063,Greene County,Alabama,8221.0,8,2.673619,10.0
5,6,21129,Lee County,Kentucky,7088.0,9,2.583054,11.0
6,7,28053,Humphreys County,Mississippi,8198.0,6,2.537897,10.0
7,8,21013,Bell County,Kentucky,26426.0,7,2.454874,11.0
8,9,28119,Quitman County,Mississippi,7038.0,6,2.433638,10.0
9,10,22107,Tensas Parish,Louisiana,4435.0,9,2.386655,10.0


Top 25 counties with population >= 50,000:


,rank,fips,county,state,pop,rucc,cbs11,n_top_quintile
0,1,37155,Robeson County,North Carolina,131656.0,4,1.807892,10.0
1,2,54055,Mercer County,West Virginia,59370.0,4,1.579230,9.0
2,3,22097,St. Landry Parish,Louisiana,82786.0,4,1.474480,8.0
3,4,21195,Pike County,Kentucky,58595.0,7,1.428913,9.0
4,5,54081,Raleigh County,West Virginia,74452.0,3,1.407870,9.0
5,6,48427,Starr County,Texas,64032.0,4,1.357027,7.0
6,7,39145,Scioto County,Ohio,75441.0,4,1.345205,9.0
7,8,37083,Halifax County,North Carolina,50678.0,4,1.335905,9.0
8,9,13095,Dougherty County,Georgia,88696.0,3,1.323702,7.0
9,10,21199,Pulaski County,Kentucky,64789.0,5,1.304504,9.0


[okn_figstyle] saved Figure 2: figures/fig2_top_counties.png


**Figure 2.** Top 20 counties by `cbs11` among all counties (A) and among counties with >= 50,000 residents (B). Bar labels = number of the 11 conditions on which the county is in the national top quintile; colour = metro (RUCC 1-3) vs. nonmetro (RUCC 4-9) from ruralkg.

*Interpretation:* the highest-burden counties are small, nonmetro counties of the Mississippi Delta, Central Appalachia and the Black Belt, plus Todd County (SD, Rosebud Reservation); almost all are in the top quintile on 9-11 of the 11 conditions, i.e. the burden is broad-based multimorbidity rather than one outlying disease.

In [18]:
mm = df["n_top_quintile"].value_counts().sort_index().rename("counties")
print("Counties by number of conditions in the national top quintile:"); print(mm.to_string())
print("\nTop 10 counties for each condition:")
top_cond = A.top_by_condition(df, 10)
display(top_cond.groupby("condition", sort=False).head(5))
states = A.state_summary(df)
display(states)

Counties by number of conditions in the national top quintile:
n_top_quintile
0.0     1363
1.0      370
2.0      245
3.0      149
4.0      148
5.0      131
6.0      130
7.0      107
8.0       96
9.0       87
10.0      81
11.0      33

Top 10 counties for each condition:


,condition,rank,fips,county,state,prevalence_%,pop
0,Diabetes,1,22035,East Carroll Parish,Louisiana,21.5,6947.0
1,Diabetes,2,48377,Presidio County,Texas,20.8,6808.0
2,Diabetes,3,48507,Zavala County,Texas,20.2,11930.0
3,Diabetes,4,46121,Todd County,South Dakota,19.9,10308.0
4,Diabetes,5,48127,Dimmit County,Texas,19.7,10232.0
10,Obesity,1,28051,Holmes County,Mississippi,53.2,17414.0
11,Obesity,2,28083,Leflore County,Mississippi,53.2,28764.0
12,Obesity,3,01131,Wilcox County,Alabama,51.3,10552.0
13,Obesity,4,22035,East Carroll Parish,Louisiana,51.2,6947.0
14,Obesity,5,01105,Perry County,Alabama,50.9,9104.0


,state,counties_scored,median_cbs11,counties_in_national_top_decile,counties_high_multimorbidity_ge8
0,Mississippi,81,1.090,41,44
1,West Virginia,50,1.075,25,21
2,Kentucky,110,0.999,49,42
3,Louisiana,61,0.899,22,24
4,Alabama,65,0.853,22,22
5,Arkansas,74,0.813,20,24
6,Tennessee,93,0.777,19,20
7,Georgia,142,0.684,30,29
8,Oklahoma,76,0.620,16,23
9,South Carolina,45,0.604,9,11


## 4. Maps

Static small multiples for the contiguous US (Figure 3) and an **interactive choropleth** (`outputs/county_disease_map.html`, a folium/Leaflet map on Esri World Street Map tiles): use the drop-down at top left to switch between the composite scores, the multimorbidity count, each of the 11 conditions and the six related outcomes. Hover shows county, state, value, population, RUCC, the number of conditions available and the composite rank. County boundaries: Census 2010 cartographic boundaries keyed by FIPS (`geo/us_counties_fips.geojson`, downloaded from plotly/datasets if absent).

In [19]:
geo = A.load_geojson()
gdf = A.counties_gdf(geo)
print("counties without a polygon:", len(set(df.fips) - set(gdf.fips)))
A.static_small_multiples(gdf, df, "figures/fig3_maps.png", 3);

counties without a polygon: 0
[okn_figstyle] saved Figure 3: figures/fig3_maps.png


**Figure 3.** County prevalence of each condition (A-K; colour scale clipped to the 2nd-98th percentile of that condition) and the composite `cbs11` (L; diverging, blue = below the county average, red = above). Grey = not available (Florida for H-L; New Jersey for C-G and L). Alaska and Hawaii are on the interactive map.

*Interpretation:* every cardiometabolic condition peaks in a band from the lower Mississippi Valley through Alabama/Georgia into Appalachia; asthma and depression have a different geography (Northeast, Pacific Northwest, Appalachia), and Native American reservation counties of the Dakotas and the Navajo Nation stand out on most maps.

In [20]:
fig_map = A.interactive_map(df, geo, "outputs/county_disease_map.html")
print("interactive map written to outputs/county_disease_map.html", f"({os.path.getsize('outputs/county_disease_map.html')/1e6:.1f} MB)")
IFrame("outputs/county_disease_map.html", width="100%", height=660)

interactive map written to outputs/county_disease_map.html (2.8 MB)


## 5. Correlation structure and confounding

* **Raw associations:** Spearman rho across the 11 conditions, two distress measures and four CVD mortality/hospitalization rates; hierarchical clustering (average linkage on 1 - rho).
* **Partial associations:** every variable is rank-transformed and regressed on (ranked) covariates; residuals are correlated (= partial Spearman). Three nested adjustment sets: (1) demographic/SES/geographic - % age 65+, % Black, % Hispanic, % poverty, % bachelor's degree, log median income, % rural; (2) + state fixed effects; (3) + smoking and physical inactivity. Set 3 removes shared behavioural pathways, which are mediators rather than pure confounders, so it is a lower bound on the residual co-occurrence.

In [21]:
cs = A.correlation_sets(df)
print("complete cases for the correlation analysis:", cs["n"])
order = A.clustered_heatmap(cs["raw"], "figures/fig4_corr_raw.png", 4,
                            "Clustered Spearman correlation across county disease measures (raw)")
A.clustered_heatmap(cs["adj_demo_state_behav"], "figures/fig5_corr_adjusted.png", 5,
                    "Partial Spearman correlation after demographic, SES, state and behavioural adjustment", order=order)
iu = np.triu_indices(len(A.COND_ALL), 1)
print("mean pairwise rho among the 11 conditions:")
for k in ["raw", "adj_demographic", "adj_demo_state", "adj_demo_state_behav"]:
    print(f"  {k:22s} {cs[k].loc[A.COND_ALL, A.COND_ALL].values[iu].mean():.3f}")
pairs = A.corr_pairs(cs)
print("\nStrongest raw pairs:"); display(pairs.head(10))
print("Pairs that hold after full adjustment:"); display(pairs.sort_values("rho_adj_full", ascending=False).head(8))
print("Pairs most explained by confounders:"); display(pairs.sort_values("attenuation_raw_to_full", ascending=False).head(8))

complete cases for the correlation analysis: 2918
[okn_figstyle] saved Figure 5: figures/fig5_corr_adjusted.png
mean pairwise rho among the 11 conditions:
  raw                    0.636
  adj_demographic        0.354
  adj_demo_state         0.310
  adj_demo_state_behav   0.205

Strongest raw pairs:


,var_a,var_b,rho_raw,rho_adj_demographic,rho_adj_demo_state,rho_adj_full,attenuation_raw_to_full
0,Diabetes,Chronic kidney disease,0.929,0.671,0.751,0.650,0.279
1,Chronic kidney disease,Stroke,0.920,0.651,0.648,0.549,0.371
2,Coronary heart disease,Stroke,0.911,0.735,0.823,0.765,0.145
3,Diabetes,Stroke,0.887,0.490,0.535,0.396,0.491
4,Hypertension,Stroke,0.875,0.595,0.727,0.656,0.219
5,COPD,Coronary heart disease,0.873,0.652,0.586,0.403,0.470
6,Diabetes,Hypertension,0.865,0.624,0.447,0.315,0.550
7,COPD,Arthritis,0.847,0.666,0.717,0.578,0.269
8,Hypertension,Coronary heart disease,0.837,0.627,0.624,0.518,0.320
9,Chronic kidney disease,Coronary heart disease,0.831,0.444,0.509,0.371,0.460


Pairs that hold after full adjustment:


,var_a,var_b,rho_raw,rho_adj_demographic,rho_adj_demo_state,rho_adj_full,attenuation_raw_to_full
2,Coronary heart disease,Stroke,0.911,0.735,0.823,0.765,0.145
4,Hypertension,Stroke,0.875,0.595,0.727,0.656,0.219
0,Diabetes,Chronic kidney disease,0.929,0.671,0.751,0.650,0.279
7,COPD,Arthritis,0.847,0.666,0.717,0.578,0.269
1,Chronic kidney disease,Stroke,0.920,0.651,0.648,0.549,0.371
8,Hypertension,Coronary heart disease,0.837,0.627,0.624,0.518,0.320
12,Asthma,Arthritis,0.799,0.590,0.596,0.478,0.322
16,Asthma,COPD,0.758,0.531,0.608,0.451,0.307


Pairs most explained by confounders:


,var_a,var_b,rho_raw,rho_adj_demographic,rho_adj_demo_state,rho_adj_full,attenuation_raw_to_full
19,Diabetes,High cholesterol,0.725,0.428,0.024,-0.085,0.810
13,High cholesterol,Hypertension,0.797,0.573,0.088,0.020,0.776
21,Diabetes,COPD,0.697,0.306,0.173,-0.063,0.759
29,Obesity,COPD,0.648,0.155,0.129,-0.048,0.696
31,Chronic kidney disease,High cholesterol,0.635,0.214,0.059,-0.051,0.686
28,High cholesterol,Stroke,0.653,0.189,0.062,-0.025,0.678
33,Obesity,High cholesterol,0.616,0.280,0.020,-0.052,0.668
10,Diabetes,Coronary heart disease,0.814,0.428,0.376,0.184,0.630


**Figure 4** (`figures/fig4_corr_raw.png`) clusters the measures into a tight **cardiometabolic block** (diabetes, CKD, stroke, CHD, hypertension, physical distress), a **respiratory/musculoskeletal block** (COPD, asthma, arthritis, mental distress), the two hospitalization rates, and depression as an outlier. **Figure 5** (`figures/fig5_corr_adjusted.png`) repeats the matrix in the same order after full adjustment.

*Interpretation:* about two thirds of the raw co-variation disappears once age, race/ethnicity, socioeconomic status, rurality, state and behaviour are accounted for (mean rho 0.64 -> 0.21). What survives is biologically coherent - CHD-stroke, hypertension-stroke, diabetes-CKD, COPD-arthritis/asthma - while e.g. diabetes-high cholesterol and high cholesterol-hypertension fall to ~0, i.e. their raw association is a shared socioeconomic and regional gradient. Depression becomes *negatively* related to diabetes after adjustment.

In [22]:
cov_assoc = A.covariate_associations(df)
A.plot_covariate_heatmap(cov_assoc, "figures/fig6_covariates.png", 6)
reg_tab, reg_r2 = A.composite_regression(df)
display(reg_tab); display(reg_r2)
A.plot_confounders(df, reg_tab, "figures/fig7_confounders.png", 7)
print(df.groupby("metro")["cbs11"].describe().round(2))

[okn_figstyle] saved Figure 6: figures/fig6_covariates.png


,covariate,beta_per_SD,ci_low,ci_high,p_value
0,% age 65+,-0.0206,-0.0336,-0.0077,0.0018
1,% Black non-Hispanic,0.0889,0.0742,0.1036,0.0000
2,% Hispanic,-0.0234,-0.0446,-0.0023,0.0296
3,% in poverty,0.0577,0.0344,0.0810,0.0000
4,% bachelor's degree,0.0564,0.0312,0.0816,0.0000
5,log median household income,-0.0740,-0.0974,-0.0505,0.0000
6,% rural population,-0.0222,-0.0313,-0.0131,0.0000
7,% adult smoking,0.3824,0.3243,0.4404,0.0000
8,% physically inactive,0.2938,0.2354,0.3522,0.0000


,model,r2,n
0,covariates,0.884,2934
1,covariates + state FE,0.970,2934
2,state FE only,0.611,2934


[okn_figstyle] saved Figure 7: figures/fig7_confounders.png
                      count  mean   std   min   25%   50%   75%   max
metro                                                                
Metro (RUCC 1-3)     1040.0 -0.19  0.71 -2.13 -0.69 -0.22  0.30  2.04
Nonmetro (RUCC 4-9)  1900.0  0.11  0.83 -2.04 -0.55  0.03  0.73  2.96


**Figure 6.** Spearman correlation of each condition with county covariates (ACS/SAIPE via AHRQ SDOH 2020, CHR 2023, RUCC from ruralkg). **Figure 7.** (A) Standardized coefficients for `cbs11` from OLS with state fixed effects and state-clustered 95% CIs; (B) `cbs11` by RUCC.

*Interpretation:* the composite tracks smoking (rho 0.86), physical inactivity (0.83), poverty (0.78) and low income/education (about -0.75); it is essentially unrelated to % age 65+ in the raw data (most inputs are model-based estimates that already reflect age structure). Covariates explain 88% of the county variance and 97% with state fixed effects, so the composite is largely a socioeconomic-behavioural gradient; the state fixed effects alone explain 61%, showing strong regional clustering. Burden rises from large metros (RUCC 1, median -0.42) to nonmetro counties, but the most remote counties (RUCC 9, Great Plains and West) are again low: rurality alone is not the driver.

In [23]:
mi = A.morans_i(gdf, df, "cbs11")
print(f"Global Moran's I (queen contiguity) = {mi['I']:.3f}, permutation p = {mi['p']:.3f}, n = {mi['n']}")
lisa = df.merge(mi["lisa"], on="fips", how="left")
print(lisa["lisa"].value_counts().to_string())
print("\nHigh-High cluster counties by state:")
print(lisa[lisa.lisa == "High-High"].groupby("state").size().sort_values(ascending=False).head(10).to_string())
td = df[df.cbs11_pctile >= 90]
south = {"01","05","10","11","12","13","21","22","24","28","37","40","45","47","48","51","54"}
print(f"\nTop-decile counties: {len(td)}, population {int(td['pop'].sum()):,}; "
      f"in Census South {td.state_fips.isin(south).mean():.0%}; nonmetro {(td.rucc >= 4).mean():.0%} "
      f"(vs {(df.dropna(subset=['cbs11']).rucc >= 4).mean():.0%} of all scored counties)")

Global Moran's I (queen contiguity) = 0.748, permutation p = 0.001, n = 2930
lisa
not significant    1844
High-High           542
Low-Low             537
Low-High              9
High-Low              8

High-High cluster counties by state:
state
Kentucky         76
Georgia          69
Mississippi      69
Arkansas         57
Tennessee        51
Alabama          47
Louisiana        43
West Virginia    39
Oklahoma         24
Missouri         18

Top-decile counties: 295, population 6,657,107; in Census South 90%; nonmetro 89% (vs 65% of all scored counties)


## 6. Export

`outputs/county_chronic_disease_results.xlsx` - one sheet per table; `outputs/county_burden_scores.csv` - the full county table with all measures, covariates and scores.

In [24]:
keep = ["fips", "county", "state", "pop", "rucc", "metro", "place_coverage", "n_places"] + A.COND_ALL + list(A.OUTCOMES) \
       + ["cbs11", "cbs11_rank", "cbs11_pctile", "cbs7", "cbs_avail", "cbs_rank", "cbs_pc1", "cbs_disease_only", "n_conditions", "n_top_quintile"] \
       + ["age65", "black", "hispanic", "poverty", "bachelor", "income", "rural", "smoking", "inactivity", "food_env"]
scores = df[keep].merge(mi["lisa"], on="fips", how="left").sort_values("cbs11", ascending=False)
scores.to_csv("outputs/county_burden_scores.csv", index=False)
methods = pd.DataFrame([
    ("Unit of analysis", "US county / county-equivalent (50 states + DC), 5-digit FIPS"),
    ("KGs", "spoke-okn v0.0.6 (PREVALENCEIN_SpL, PREVALENCE_DpL, PARTOF_LpL); ruralkg v0.2.10 (RUCC)"),
    ("Tier A", "native county estimates: CHR 2023 (diabetes, obesity); AHRQ SDOH 2019 CDC PLACES county (asthma, COPD, arthritis, CKD, high cholesterol)"),
    ("Tier B", "CDC PLACES place-level age-adjusted prevalence, population-weighted roll-up to county (hypertension, CHD, stroke, depression); used only if places cover >= 10% of county population"),
    ("cbs11", "mean z-score over 11 conditions, equal weights, unweighted by population; requires all 11"),
    ("Sensitivity scores", "cbs7, cbs_avail (>=6 available), cbs_rank, cbs_pc1, cbs_disease_only"),
    ("Multimorbidity", "n_top_quintile = conditions with county >= national 80th percentile"),
    ("Correlation", "Spearman; partial = Pearson of rank residuals on covariates (+ state FE, + smoking/inactivity)"),
    ("Regression", "OLS cbs11 ~ standardized covariates + state FE, SE clustered by state"),
    ("Spatial", "Global Moran's I and LISA (queen contiguity, row-standardized, permutation p)"),
    ("Abbreviations", "CHR County Health Rankings; AHRQ SDOH Agency for Healthcare Research and Quality Social Determinants of Health database; PLACES CDC small-area estimates; RUCC Rural-Urban Continuum Code; CHD coronary heart disease; CKD chronic kidney disease; COPD chronic obstructive pulmonary disease; FE fixed effects; LISA local indicators of spatial association"),
], columns=["item", "description"])
sheets = {
    "Methods & Rules": methods, "Measure catalogue": catalogue, "County scores": scores,
    "Top 25 composite": top_all, "Top 25 pop>=50k": top_big, "Top 10 by condition": top_cond,
    "State summary": states, "Validation place vs native": validation, "Data gaps": gaps, "QC": qc,
    "Score agreement": agreement, "Corr raw": cs["raw"].rename(index=A.LABEL, columns=A.LABEL).reset_index(),
    "Corr adjusted full": cs["adj_demo_state_behav"].rename(index=A.LABEL, columns=A.LABEL).reset_index(),
    "Corr pairs": pairs, "Covariate associations": cov_assoc.rename(index=A.LABEL, columns=A.LABEL).reset_index(),
    "Composite regression": reg_tab, "Regression R2": reg_r2,
}
A.write_workbook("outputs/county_chronic_disease_results.xlsx", sheets)
print("wrote outputs/county_chronic_disease_results.xlsx with", len(sheets), "sheets and outputs/county_burden_scores.csv")

wrote outputs/county_chronic_disease_results.xlsx with 17 sheets and outputs/county_burden_scores.csv


## 7. Key findings

1. **Data.** The OKN federation supports county-level estimates for 11 chronic conditions: 7 natively (spoke-okn: County Health Rankings 2023; AHRQ SDOH / CDC PLACES 2019) and 4 - hypertension, coronary heart disease, stroke, depression - by rolling CDC PLACES place-level estimates up to counties via spoke-okn's place-to-county links. The roll-up was validated against the native estimates (r = 0.77-0.94). An 11-condition composite could be computed for **2,940 of 3,131 counties**; a coverage-maximising variant covers 3,130.
2. **Highest composite burden:** East Carroll Parish (LA), Holmes (MS), Todd (SD), McDowell (WV), Greene (AL), Lee (KY), Humphreys (MS), Bell (KY), Quitman (MS) and Tensas Parish (LA). Among counties >= 50,000 residents: Robeson (NC), Mercer (WV), St. Landry Parish (LA), Pike (KY) and Raleigh (WV).
3. **Multimorbidity:** 297 counties are in the national top quintile on >= 8 of the 11 conditions and 33 on all 11. The 295 top-decile counties (6.7 million residents) are 90% in the Census South and 89% nonmetro (vs. 65% of all scored counties). Mississippi, West Virginia, Kentucky, Louisiana and Alabama have the highest state medians.
4. **Individual-condition peaks** differ in geography: diabetes (East Carroll LA 21.5%, Presidio and Zavala TX), obesity and hypertension (Mississippi Delta, >53%), COPD, high cholesterol and depression (eastern Kentucky), arthritis (southern West Virginia), CHD/stroke/CKD/asthma (Todd County SD and other reservation counties).
5. **Robustness:** the composite is insensitive to its construction (Spearman >= 0.98 with all five variants; PC1 explains 66% of variance with near-equal loadings except depression).
6. **Spatial clustering** is strong (Moran's I = 0.75, p = 0.001): 542 counties form High-High clusters, mostly in Kentucky, Georgia, Mississippi, Arkansas, Tennessee, Alabama, Louisiana and West Virginia.
7. **Correlations:** conditions are highly co-located (mean pairwise rho 0.64), forming a cardiometabolic cluster and a respiratory/musculoskeletal cluster, with depression the least aligned. After demographic, SES, state and behavioural adjustment the mean falls to 0.21; CHD-stroke (0.77), hypertension-stroke (0.66), diabetes-CKD (0.65) and COPD-arthritis (0.58) persist, while diabetes-high cholesterol and high cholesterol-hypertension vanish.
8. **Confounders:** smoking, physical inactivity, poverty, low income and low educational attainment explain most of the composite's variation (R2 = 0.88; 0.97 with state fixed effects). The age structure of the county barely matters at this level, and the burden is highest in nonmetro counties adjacent to metros (RUCC 6-8), not in the most remote ones.

## 8. Assumptions and limitations

1. **Model-based estimates.** CHR and PLACES prevalences are small-area model estimates (BRFSS + multilevel regression and post-stratification on county demographics). They are smoothed toward demographic expectations, which (a) understates true between-county variation in small counties and (b) *inflates* correlations between conditions and with demographic covariates, because the same covariates feed every model. Partial correlations and R2 values should be read with this in mind.
2. **Mixed years and releases.** CHR 2023 release (underlying data ~2020-2021), AHRQ SDOH 2019/2020, CDC Atlas 2018/2019; the PLACES place-level release year is **not recorded** in spoke-okn. Age-adjusted vs. crude status is recorded only for the place-level data (age-adjusted). Scores are therefore built from z-scores/ranks, not absolute differences.
3. **Roll-up approximation (Tier B).** Places (incorporated cities, towns, CDPs) do not cover unincorporated areas; county values are the population-weighted mean of places, which over-represents town populations (median place coverage 50%). A place spanning several counties contributes its population equally to each. Validation against native estimates (Figure 1) supports ranking use, with a positive offset for obesity and high cholesterol.
4. **Coverage gaps in the KG.** New Jersey has no 2019 PLACES county values; Florida has no PLACES place rows (so no hypertension, CHD, stroke or depression); 22 Virginia independent cities and ~80 other counties lack place links. These counties are absent from `cbs11` (they appear in `cbs_avail`/`cbs7`). spoke-okn's IHME and WHO data are country-level only. Missing from the federation at county level: cancer incidence, heart failure, Alzheimer's/dementia, chronic liver disease and childhood asthma.
5. **Ontology mapping.** The PLACES 'High Cholesterol' measure is stored on DOID:2349 (*arteriosclerosis*) and 'Coronary Heart Disease' on DOID:3393 (*coronary artery disease*); the measures are interpreted by their survey item (`short_question_text`, q02), not by the DOID label.
6. **Composite weighting.** Equal weights across conditions, counties unweighted by population; the score measures relative prevalence burden, not case counts or disease severity, and includes risk-factor conditions (obesity, high cholesterol, hypertension) - see the `cbs_disease_only` sensitivity.
7. **Ecological inference.** All associations are between county aggregates and must not be interpreted as individual-level risk; smoking and inactivity are partly mediators, so the fully adjusted correlations are a lower bound.
8. **Boundaries.** County polygons are 2010 Census cartographic boundaries; Connecticut uses legacy counties (consistent with the KG), and recent Alaska borough changes are not reflected.

## Figures

![Figure 1. Place roll-ups vs. native county estimates for the five measures available both ways (A-E; colour = place coverage) and the distribution of place coverage (F; dashed line = 10% inclusion threshold).](figures/fig1_validation.png)

*Figure 1. Place roll-ups vs. native county estimates for the five measures available both ways (A-E; colour = place coverage) and the distribution of place coverage (F; dashed line = 10% inclusion threshold).*

![Figure 2. Top 20 counties by the 11-condition composite burden score, all counties (A) and counties with population >= 50,000 (B). Labels give the number of conditions on which the county is in the national top quintile; colour = RUCC metro/nonmetro (ruralkg).](figures/fig2_top_counties.png)

*Figure 2. Top 20 counties by the 11-condition composite burden score, all counties (A) and counties with population >= 50,000 (B). Labels give the number of conditions on which the county is in the national top quintile; colour = RUCC metro/nonmetro (ruralkg).*

![Figure 3. Contiguous-US county prevalence of the 11 conditions (A-K, colour clipped to the 2nd-98th percentile) and the composite burden score (L). Grey = not available.](figures/fig3_maps.png)

*Figure 3. Contiguous-US county prevalence of the 11 conditions (A-K, colour clipped to the 2nd-98th percentile) and the composite burden score (L). Grey = not available.*

![Figure 4. Clustered Spearman correlation (average linkage on 1 - rho) across the 11 conditions, two distress measures and four CVD mortality/hospitalization rates.](figures/fig4_corr_raw.png)

*Figure 4. Clustered Spearman correlation (average linkage on 1 - rho) across the 11 conditions, two distress measures and four CVD mortality/hospitalization rates.*

![Figure 5. Partial Spearman correlations (same order as Figure 4) after adjusting for age, race/ethnicity, poverty, education, income, rurality, state fixed effects, smoking and physical inactivity.](figures/fig5_corr_adjusted.png)

*Figure 5. Partial Spearman correlations (same order as Figure 4) after adjusting for age, race/ethnicity, poverty, education, income, rurality, state fixed effects, smoking and physical inactivity.*

![Figure 6. Spearman correlation of each condition and the composite with county covariates.](figures/fig6_covariates.png)

*Figure 6. Spearman correlation of each condition and the composite with county covariates.*

![Figure 7. (A) Adjusted associations of covariates with the composite (OLS, state fixed effects, state-clustered 95% CI). (B) Composite by Rural-Urban Continuum Code from ruralkg.](figures/fig7_confounders.png)

*Figure 7. (A) Adjusted associations of covariates with the composite (OLS, state fixed effects, state-clustered 95% CI). (B) Composite by Rural-Urban Continuum Code from ruralkg.*

## Verify against the live graphs

In [25]:
verify_all()

,query,graphs,logged_rows,live_rows,status
0,q01,spoke-okn,2,2,ok
1,q02,spoke-okn,9,9,ok
2,q03,spoke-okn,24,24,ok
3,q04,"spoke-okn, ruralkg",544,544,ok
4,q05,"spoke-okn, ruralkg",936,936,ok
5,q06,"spoke-okn, ruralkg",723,723,ok
6,q07,"spoke-okn, ruralkg",928,928,ok
7,q08,spoke-okn,476,476,ok
8,q09,spoke-okn,936,936,ok
9,q10,spoke-okn,718,718,ok
